In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# =========================
# 1. 设置文件路径
# =========================

period1_file = "sample1_200_en.xlsx"
period2_file = "alibaba_sentiment_merged.xlsx"

output_dir = Path("step1_cleaned_outputs")
output_dir.mkdir(exist_ok=True)

# =========================
# 2. 通用清洗函数
# =========================

def clean_text(x):
    """清洗文本：去除多余空格、换行，并统一为空字符串"""
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()


def parse_date_column(series):
    """统一日期格式"""
    return pd.to_datetime(series, errors="coerce")


def normalize_sentiment_score(series):
    """把 sentiment_score 转成数值型"""
    return pd.to_numeric(series, errors="coerce")


def mark_relevance(row):
    """
    根据 title/body/reason 粗略判断新闻是否与 Alibaba 相关。
    这是规则判断，不是最终人工结论。
    后面建议人工抽查。
    """
    text = " ".join([
        clean_text(row.get("title_en", "")),
        clean_text(row.get("body_en", "")),
        clean_text(row.get("sentiment_reason", ""))
    ]).lower()

    # 明确不相关的表达
    unrelated_patterns = [
        "not related to alibaba",
        "unrelated to alibaba",
        "irrelevant to alibaba",
        "not directly related",
        "does not mention alibaba",
        "no direct impact",
        "unrelated to alibaba company entities",
        "not related to alibaba group",
        "irrelevant to alibaba group",
    ]

    for p in unrelated_patterns:
        if p in text:
            return 0

    # Alibaba 及其相关业务关键词
    alibaba_keywords = [
        "alibaba",
        "baba",
        "9988.hk",
        "taobao",
        "tmall",
        "aliexpress",
        "ant group",
        "alipay",
        "aliyun",
        "ali cloud",
        "alibaba cloud",
        "cainiao",
        "ele.me",
        "lazada",
        "freshippo",
        "youku",
        "dingding",
        "dingtalk",
        "1688.com",
    ]

    for kw in alibaba_keywords:
        if kw in text:
            return 1

    return 0


def standardize_dataset(df, source_period):
    """
    对单个新闻周期进行标准化清洗。
    输出统一字段：
    date, title_en, body_en, sentiment_score, sentiment_reason,
    source, url, ticker, source_period, full_text, relevance_flag
    """

    df = df.copy()

    # -------------------------
    # 统一列名
    # -------------------------
    rename_map = {}

    # 日期列
    if "date" in df.columns:
        rename_map["date"] = "date"
    elif "inserted_at" in df.columns:
        rename_map["inserted_at"] = "date"
    elif "published_at" in df.columns:
        rename_map["published_at"] = "date"
    elif "time" in df.columns:
        rename_map["time"] = "date"

    # 标题列
    if "title_en" in df.columns:
        rename_map["title_en"] = "title_en"
    elif "original_title_en" in df.columns:
        rename_map["original_title_en"] = "title_en"
    elif "title" in df.columns:
        rename_map["title"] = "title_en"

    # 正文列
    if "body_en" in df.columns:
        rename_map["body_en"] = "body_en"
    elif "original_body_en" in df.columns:
        rename_map["original_body_en"] = "body_en"
    elif "body" in df.columns:
        rename_map["body"] = "body_en"
    elif "content" in df.columns:
        rename_map["content"] = "body_en"

    # 情绪分数列
    if "sentiment_score" in df.columns:
        rename_map["sentiment_score"] = "sentiment_score"
    elif "score" in df.columns:
        rename_map["score"] = "sentiment_score"

    # 情绪理由列
    if "sentiment_reason" in df.columns:
        rename_map["sentiment_reason"] = "sentiment_reason"
    elif "reason" in df.columns:
        rename_map["reason"] = "sentiment_reason"

    # 来源列
    if "source" in df.columns:
        rename_map["source"] = "source"
    elif "publisher" in df.columns:
        rename_map["publisher"] = "source"
    elif "newspaper" in df.columns:
        rename_map["newspaper"] = "source"

    # URL列
    if "url" in df.columns:
        rename_map["url"] = "url"
    elif "link" in df.columns:
        rename_map["link"] = "url"

    # ticker列
    if "ticker" in df.columns:
        rename_map["ticker"] = "ticker"

    df = df.rename(columns=rename_map)

    # -------------------------
    # 如果某些列不存在，则补空
    # -------------------------
    required_columns = [
        "date",
        "title_en",
        "body_en",
        "sentiment_score",
        "sentiment_reason",
        "source",
        "url",
        "ticker",
    ]

    for col in required_columns:
        if col not in df.columns:
            df[col] = np.nan

    # -------------------------
    # 基础清洗
    # -------------------------
    df["date"] = parse_date_column(df["date"])
    df["title_en"] = df["title_en"].apply(clean_text)
    df["body_en"] = df["body_en"].apply(clean_text)
    df["sentiment_reason"] = df["sentiment_reason"].apply(clean_text)
    df["source"] = df["source"].apply(clean_text)
    df["url"] = df["url"].apply(clean_text)

    # ticker 默认填 9988.HK
    df["ticker"] = df["ticker"].apply(clean_text)
    df.loc[df["ticker"] == "", "ticker"] = "9988.HK"

    # sentiment_score 转数值
    df["sentiment_score"] = normalize_sentiment_score(df["sentiment_score"])

    # 新增 full_text
    df["full_text"] = (
        df["title_en"].fillna("") + ". " + df["body_en"].fillna("")
    ).apply(clean_text)

    # 新增 source_period
    df["source_period"] = source_period

    # 新增 article_id
    df = df.reset_index(drop=True)
    df["article_id"] = [
        f"{source_period}_{i+1:04d}" for i in range(len(df))
    ]

    # 标记 relevance_flag
    df["relevance_flag"] = df.apply(mark_relevance, axis=1)

    # -------------------------
    # 去重：基于 title + body
    # -------------------------
    before_dedup = len(df)

    df["dedup_key"] = (
        df["title_en"].str.lower().fillna("") + " " +
        df["body_en"].str.lower().fillna("")
    )

    df = df.drop_duplicates(subset=["dedup_key"], keep="first").copy()
    after_dedup = len(df)

    df = df.drop(columns=["dedup_key"])

    # -------------------------
    # 按日期排序
    # -------------------------
    df = df.sort_values("date").reset_index(drop=True)

    # -------------------------
    # 生成质量检查信息
    # -------------------------
    quality = {
        "source_period": source_period,
        "raw_rows": before_dedup,
        "rows_after_dedup": after_dedup,
        "duplicates_removed": before_dedup - after_dedup,
        "missing_date": df["date"].isna().sum(),
        "missing_title_en": (df["title_en"] == "").sum(),
        "missing_body_en": (df["body_en"] == "").sum(),
        "missing_sentiment_score": df["sentiment_score"].isna().sum(),
        "missing_sentiment_reason": (df["sentiment_reason"] == "").sum(),
        "invalid_sentiment_score": (~df["sentiment_score"].between(1, 5)).sum(),
        "relevance_flag_1": (df["relevance_flag"] == 1).sum(),
        "relevance_flag_0": (df["relevance_flag"] == 0).sum(),
        "min_date": df["date"].min(),
        "max_date": df["date"].max(),
    }

    # -------------------------
    # 只保留最终需要字段
    # -------------------------
    final_columns = [
        "article_id",
        "date",
        "title_en",
        "body_en",
        "full_text",
        "sentiment_score",
        "sentiment_reason",
        "source",
        "url",
        "ticker",
        "source_period",
        "relevance_flag",
    ]

    df = df[final_columns]

    return df, quality


# =========================
# 3. 读取两个 Excel
# =========================

period1_raw = pd.read_excel(period1_file)
period2_raw = pd.read_excel(period2_file)

print("Period 1 原始列名：")
print(period1_raw.columns.tolist())

print("\nPeriod 2 原始列名：")
print(period2_raw.columns.tolist())


# =========================
# 4. 分别清洗两个周期
# =========================

period1_cleaned, quality1 = standardize_dataset(
    period1_raw,
    source_period="period_1_2025"
)

period2_cleaned, quality2 = standardize_dataset(
    period2_raw,
    source_period="period_2_2026"
)


# =========================
# 5. 输出质量检查报告
# =========================

quality_report = pd.DataFrame([quality1, quality2])

score_distribution = (
    pd.concat([period1_cleaned, period2_cleaned], ignore_index=True)
    .groupby(["source_period", "sentiment_score"])
    .size()
    .reset_index(name="count")
)

print("\n========== 清洗质量报告 ==========")
display(quality_report)

print("\n========== 情绪分数分布 ==========")
display(score_distribution)


# =========================
# 6. 检查异常样本
# =========================

def show_problem_rows(df, name):
    print(f"\n========== {name} 异常样本检查 ==========")

    problem_rows = df[
        df["date"].isna()
        | (df["title_en"] == "")
        | (df["body_en"] == "")
        | df["sentiment_score"].isna()
        | (~df["sentiment_score"].between(1, 5))
        | (df["sentiment_reason"] == "")
    ]

    print(f"{name} 异常样本数量：{len(problem_rows)}")

    if len(problem_rows) > 0:
        display(problem_rows[
            [
                "article_id",
                "date",
                "title_en",
                "sentiment_score",
                "sentiment_reason",
                "source_period",
                "relevance_flag"
            ]
        ].head(20))

show_problem_rows(period1_cleaned, "Period 1")
show_problem_rows(period2_cleaned, "Period 2")


# =========================
# 7. 保存清洗后的结果
# =========================

period1_cleaned.to_csv(output_dir / "period_1_2025_cleaned.csv", index=False, encoding="utf-8-sig")
period2_cleaned.to_csv(output_dir / "period_2_2026_cleaned.csv", index=False, encoding="utf-8-sig")
quality_report.to_csv(output_dir / "step1_quality_report.csv", index=False, encoding="utf-8-sig")
score_distribution.to_csv(output_dir / "step1_score_distribution.csv", index=False, encoding="utf-8-sig")

with pd.ExcelWriter(output_dir / "step1_cleaned_news_outputs.xlsx", engine="openpyxl") as writer:
    period1_cleaned.to_excel(writer, sheet_name="period_1_2025_cleaned", index=False)
    period2_cleaned.to_excel(writer, sheet_name="period_2_2026_cleaned", index=False)
    quality_report.to_excel(writer, sheet_name="quality_report", index=False)
    score_distribution.to_excel(writer, sheet_name="score_distribution", index=False)

print("\n清洗完成，文件已保存到：", output_dir.resolve())


# =========================
# 8. 查看清洗后的前几行
# =========================

print("\n========== Period 1 cleaned preview ==========")
display(period1_cleaned.head())

print("\n========== Period 2 cleaned preview ==========")
display(period2_cleaned.head())

Period 1 原始列名：
['inserted_at', 'title_en', 'body_en', 'newspaper', 'sentiment_score', 'sentiment_reason']

Period 2 原始列名：
['company_name', 'ticker', 'sector', 'article_id', 'date', 'url', 'publisher', 'original_title_en', 'original_body_en', 'sentiment_score', 'sentiment_reason']

========== 清洗质量报告 ==========


,source_period,raw_rows,rows_after_dedup,duplicates_removed,missing_date,missing_title_en,missing_body_en,missing_sentiment_score,missing_sentiment_reason,invalid_sentiment_score,relevance_flag_1,relevance_flag_0,min_date,max_date
0,period_1_2025,200,200,0,0,0,0,0,0,0,41,159,2025-06-03 06:02:57,2025-07-01 07:17:16
1,period_2_2026,176,176,0,6,0,0,0,0,0,113,63,2026-02-05 00:00:00,2026-04-23 18:02:00



========== 情绪分数分布 ==========


,source_period,sentiment_score,count
0,period_1_2025,2,1
1,period_1_2025,3,167
2,period_1_2025,4,29
3,period_1_2025,5,3
4,period_2_2026,1,2
5,period_2_2026,2,50
6,period_2_2026,3,69
7,period_2_2026,4,49
8,period_2_2026,5,6



========== Period 1 异常样本检查 ==========
Period 1 异常样本数量：0

========== Period 2 异常样本检查 ==========
Period 2 异常样本数量：6


,article_id,date,title_en,sentiment_score,sentiment_reason,source_period,relevance_flag
170,period_2_2026_0064,NaT,url https://www.yicai.com/news/103088709.html ...,5,"Alibaba has established Alibaba Token Hub, cre...",period_2_2026,1
171,period_2_2026_0066,NaT,url https://www.yicai.com/news/103088550.html ...,4,Alibaba’s Token Hub remains the clearest posit...,period_2_2026,1
172,period_2_2026_0070,NaT,url https://www.yicai.com/news/103088893.html ...,4,"The news mix is mostly macro, but Alibaba’s To...",period_2_2026,1
173,period_2_2026_0117,NaT,State Administration for Market Regulation 202...,2,The annual report confirms sustained high-pres...,period_2_2026,0
174,period_2_2026_0118,NaT,"Starting in April, these new regulations will ...",3,A mix of new rules brings both platform constr...,period_2_2026,1
175,period_2_2026_0119,NaT,The State Administration for Market Regulation...,2,tricter online food regulation raises complian...,period_2_2026,0



清洗完成，文件已保存到： /Users/sunny/Desktop/Capstone/中期汇报/step1_cleaned_outputs

========== Period 1 cleaned preview ==========


,article_id,date,title_en,body_en,full_text,sentiment_score,sentiment_reason,source,url,ticker,source_period,relevance_flag
0,period_1_2025_0200,2025-06-03 06:02:57,Platform economy stimulates the vitality of sm...,Individual industrial and commercial household...,Platform economy stimulates the vitality of sm...,3,Unrelated to Alibaba company entities,Economic Daily,,9988.HK,period_1_2025,0
1,period_1_2025_0199,2025-06-03 08:01:43,"""Underage movie viewing discount policy 'hide ...","Behind a regular movie ticket, there is a hidd...","""Underage movie viewing discount policy 'hide ...",3,Unrelated to Alibaba company entities,Procuratorate daily,,9988.HK,period_1_2025,0
2,period_1_2025_0198,2025-06-03 08:02:03,Setting up a scam in the name of borrowing a h...,"Self-proclaimed as the ""red envelope snatching...",Setting up a scam in the name of borrowing a h...,3,Unrelated to Alibaba company entities,Procuratorate daily,,9988.HK,period_1_2025,0
3,period_1_2025_0197,2025-06-03 08:23:36,Notice on holding the 2024 Academic Annual Con...,Relevant units: Pharmaceutical excipients are ...,Notice on holding the 2024 Academic Annual Con...,3,Unrelated to Alibaba company entities,ChinaNationalPharmaceuticalPackagingAssociation,,9988.HK,period_1_2025,0
4,period_1_2025_0196,2025-06-03 14:13:17,"Seize the 90-day window period ""Foreign Trade ...",China Business News reporter Li Li reported fr...,"Seize the 90-day window period ""Foreign Trade ...",4,"Taobao's 618 event is being promoted globally,...",China Business Journal,,9988.HK,period_1_2025,1



========== Period 2 cleaned preview ==========


,article_id,date,title_en,body_en,full_text,sentiment_score,sentiment_reason,source,url,ticker,source_period,relevance_flag
0,period_2_2026_0001,2026-02-05 00:00:00,"New Regulations Target Big Data ""Price Discrim...","Recently, the State Administration for Market ...","New Regulations Target Big Data ""Price Discrim...",2,"New regulations have been introduced, cutting ...",People's Post & Telecom,https://proxy.scrapeops.io/v1/?api_key=164c487...,9988.HK,period_2_2026,1
1,period_2_2026_0002,2026-02-05 00:00:00,"Meituan Buys Dingdong Maicai, Defense or Attack?","On February 5th, Meituan announced in a filing...","Meituan Buys Dingdong Maicai, Defense or Attac...",3,Alibaba is developing its fresh produce instan...,yicai,https://www.yicai.com/news/103041444.html,9988.HK,period_2_2026,1
2,period_2_2026_0003,2026-02-05 14:33:00,State Council Information Office Holds Press C...,The State Council Information Office held a pr...,State Council Information Office Holds Press C...,2,Tighter platform regulation may limit Alibaba’...,StateAdministrationForMarketRegulation,https://www.samr.gov.cn/xw/xwfbt/art/2026/art_...,9988.HK,period_2_2026,1
3,period_2_2026_0004,2026-02-05 22:14:00,International Olympic Committee Officially Ann...,Original Title: IOC Officially Announces at Mi...,International Olympic Committee Officially Ann...,5,The IOC deal strongly boosts Alibaba’s global ...,sina_news,https://mil.news.sina.com.cn/2026-02-05/doc-in...,9988.HK,period_2_2026,1
4,period_2_2026_0008,2026-02-12 00:00:00,"From ""Data Governance"" to ""Data Intelligence""",With the advent of the large model democratiza...,"From ""Data Governance"" to ""Data Intelligence""....",3,"The trend supports Alibaba’s AI strategy, but ...",People's Post & Telecom,https://proxy.scrapeops.io/v1/?api_key=164c487...,9988.HK,period_2_2026,0


In [4]:
# 如果你还没有安装 yfinance，先运行这一行
!pip install yfinance openpyxl -q

In [7]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path

# =========================
# 1. 设置输入输出路径
# =========================

input_dir = Path("step1_cleaned_outputs")

period1_file = input_dir / "period_1_2025_cleaned.csv"
period2_file = input_dir / "period_2_2026_cleaned.csv"

output_dir = Path("step2_step3_price_return_outputs")
output_dir.mkdir(exist_ok=True)

# 主股票和市场指数
stock_ticker = "9988.HK"
market_ticker = "3032.HK"

# 需要计算的未来收益窗口
horizons = [1, 5, 21]


# =========================
# 2. 读取 Step 1 清洗后的新闻数据
# =========================

period1 = pd.read_csv(period1_file)
period2 = pd.read_csv(period2_file)

period1["date"] = pd.to_datetime(period1["date"], errors="coerce")
period2["date"] = pd.to_datetime(period2["date"], errors="coerce")

print("Period 1 日期范围：", period1["date"].min(), "到", period1["date"].max())
print("Period 2 日期范围：", period2["date"].min(), "到", period2["date"].max())


# =========================
# 3. 确定股价下载时间范围
# =========================

all_news_dates = pd.concat([period1["date"], period2["date"]]).dropna()

start_date = all_news_dates.min() - pd.Timedelta(days=15)

# 为了计算未来 21 个交易日收益，结束日期需要往后延长
# 如果当前日期还不够远，yfinance 会自动只下载到最新可获得交易日
end_date = all_news_dates.max() + pd.Timedelta(days=60)

print("计划下载股价范围：", start_date.date(), "到", end_date.date())


# =========================
# 4. 下载股价数据
# =========================

def download_price_data(ticker, start, end):
    """
    下载日频股价数据。
    auto_adjust=False 是为了保留 Adj Close。
    """
    data = yf.download(
        ticker,
        start=start.strftime("%Y-%m-%d"),
        end=end.strftime("%Y-%m-%d"),
        auto_adjust=False,
        progress=False
    )

    if data.empty:
        raise ValueError(f"{ticker} 下载失败，返回为空。请检查 ticker 是否正确。")

    # 如果 yfinance 返回多层列名，这里压平
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)

    data = data.reset_index()

    # 统一列名
    data = data.rename(columns={
        "Date": "price_date",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Adj Close": "adj_close",
        "Volume": "volume"
    })

    data["price_date"] = pd.to_datetime(data["price_date"])
    data = data.sort_values("price_date").reset_index(drop=True)

    # 如果没有 adj_close，就用 close 代替
    if "adj_close" not in data.columns or data["adj_close"].isna().all():
        data["adj_close"] = data["close"]

    data["ticker"] = ticker

    return data


stock_price = download_price_data(stock_ticker, start_date, end_date)
market_price = download_price_data(market_ticker, start_date, end_date)

print("\n9988.HK 股价数据：")
display(stock_price.head())
display(stock_price.tail())

print("\nHSTECH.HK 指数数据：")
display(market_price.head())
display(market_price.tail())


# =========================
# 5. 检查下载结果
# =========================

print("9988.HK 数据日期范围：", stock_price["price_date"].min(), "到", stock_price["price_date"].max())
print("HSTECH.HK 数据日期范围：", market_price["price_date"].min(), "到", market_price["price_date"].max())

print("9988.HK 行数：", len(stock_price))
print("HSTECH.HK 行数：", len(market_price))


# =========================
# 6. 计算某个价格表的 forward return
# =========================

def add_forward_returns(price_df, horizons=[1, 5, 21], prefix="ret"):
    """
    对价格数据计算未来收益。
    ret_h = adj_close[t+h] / adj_close[t] - 1
    """
    df = price_df.copy()
    df = df.sort_values("price_date").reset_index(drop=True)

    for h in horizons:
        df[f"{prefix}_{h}d"] = df["adj_close"].shift(-h) / df["adj_close"] - 1

    return df


stock_price_ret = add_forward_returns(stock_price, horizons=horizons, prefix="stock_ret")
market_price_ret = add_forward_returns(market_price, horizons=horizons, prefix="market_ret")


# =========================
# 7. 给新闻匹配 event_date 和 forward return
# =========================

def attach_returns_to_news(news_df, stock_ret_df, market_ret_df, horizons=[1, 5, 21]):
    """
    对每条新闻：
    1. 将新闻 date 映射到下一个可交易日 event_date
    2. 匹配 9988.HK 的未来收益
    3. 匹配 HSTECH 的未来收益
    4. 计算 abnormal return
    """

    news = news_df.copy()
    news["date"] = pd.to_datetime(news["date"], errors="coerce")
    news["news_date_only"] = news["date"].dt.normalize()

    stock = stock_ret_df.copy()
    market = market_ret_df.copy()

    stock["price_date"] = pd.to_datetime(stock["price_date"])
    market["price_date"] = pd.to_datetime(market["price_date"])

    stock_dates = stock["price_date"].sort_values().reset_index(drop=True)

    def get_next_trading_day(news_date):
        """
        如果新闻日期是交易日，则用当天；
        如果新闻日期不是交易日，则用下一个交易日。
        """
        if pd.isna(news_date):
            return pd.NaT

        possible_dates = stock_dates[stock_dates >= news_date]
        if len(possible_dates) == 0:
            return pd.NaT

        return possible_dates.iloc[0]

    news["event_date"] = news["news_date_only"].apply(get_next_trading_day)

    # 只保留需要合并的股价收益列
    stock_cols = ["price_date", "adj_close"] + [f"stock_ret_{h}d" for h in horizons]
    market_cols = ["price_date"] + [f"market_ret_{h}d" for h in horizons]

    stock_merge = stock[stock_cols].rename(columns={
        "price_date": "event_date",
        "adj_close": "stock_event_adj_close"
    })

    market_merge = market[market_cols].rename(columns={
        "price_date": "event_date"
    })

    news = news.merge(stock_merge, on="event_date", how="left")
    news = news.merge(market_merge, on="event_date", how="left")

    # 改名：stock_ret_1d -> ret_1d
    for h in horizons:
        news[f"ret_{h}d"] = news[f"stock_ret_{h}d"]
        news[f"hstech_ret_{h}d"] = news[f"market_ret_{h}d"]
        news[f"abret_{h}d"] = news[f"ret_{h}d"] - news[f"hstech_ret_{h}d"]

    # 删除中间列
    drop_cols = []
    for h in horizons:
        drop_cols.extend([f"stock_ret_{h}d", f"market_ret_{h}d"])

    news = news.drop(columns=drop_cols)

    # 按日期排序
    news = news.sort_values(["event_date", "date"]).reset_index(drop=True)

    return news


period1_with_returns = attach_returns_to_news(
    period1,
    stock_price_ret,
    market_price_ret,
    horizons=horizons
)

period2_with_returns = attach_returns_to_news(
    period2,
    stock_price_ret,
    market_price_ret,
    horizons=horizons
)


# =========================
# 8. 生成收益匹配质量报告
# =========================

def return_quality_report(df, name, horizons=[1, 5, 21]):
    report = {
        "source_period": name,
        "rows": len(df),
        "missing_event_date": df["event_date"].isna().sum(),
        "missing_stock_event_adj_close": df["stock_event_adj_close"].isna().sum(),
    }

    for h in horizons:
        report[f"missing_ret_{h}d"] = df[f"ret_{h}d"].isna().sum()
        report[f"missing_hstech_ret_{h}d"] = df[f"hstech_ret_{h}d"].isna().sum()
        report[f"missing_abret_{h}d"] = df[f"abret_{h}d"].isna().sum()

    return report


quality_returns = pd.DataFrame([
    return_quality_report(period1_with_returns, "period_1_2025", horizons),
    return_quality_report(period2_with_returns, "period_2_2026", horizons)
])

print("\n========== 收益匹配质量报告 ==========")
display(quality_returns)


# =========================
# 9. 查看收益列是否正常
# =========================

return_cols = [
    "article_id",
    "date",
    "event_date",
    "sentiment_score",
    "source_period",
    "relevance_flag",
    "stock_event_adj_close",
    "ret_1d",
    "ret_5d",
    "ret_21d",
    "hstech_ret_1d",
    "hstech_ret_5d",
    "hstech_ret_21d",
    "abret_1d",
    "abret_5d",
    "abret_21d",
]

print("\n========== Period 1 with returns preview ==========")
display(period1_with_returns[return_cols].head())

print("\n========== Period 2 with returns preview ==========")
display(period2_with_returns[return_cols].head())


# =========================
# 10. 保存 Step 2 + Step 3 输出
# =========================

stock_price_ret.to_csv(output_dir / "9988_HK_price_with_forward_returns.csv", index=False, encoding="utf-8-sig")
market_price_ret.to_csv(output_dir / "HSTECH_HK_price_with_forward_returns.csv", index=False, encoding="utf-8-sig")

period1_with_returns.to_csv(output_dir / "period_1_2025_with_returns.csv", index=False, encoding="utf-8-sig")
period2_with_returns.to_csv(output_dir / "period_2_2026_with_returns.csv", index=False, encoding="utf-8-sig")
quality_returns.to_csv(output_dir / "step2_step3_return_quality_report.csv", index=False, encoding="utf-8-sig")

with pd.ExcelWriter(output_dir / "step2_step3_price_return_outputs.xlsx", engine="openpyxl") as writer:
    stock_price_ret.to_excel(writer, sheet_name="9988_HK_price", index=False)
    market_price_ret.to_excel(writer, sheet_name="HSTECH_HK_price", index=False)
    period1_with_returns.to_excel(writer, sheet_name="period_1_with_returns", index=False)
    period2_with_returns.to_excel(writer, sheet_name="period_2_with_returns", index=False)
    quality_returns.to_excel(writer, sheet_name="quality_report", index=False)

print("\nStep 2 + Step 3 完成，文件已保存到：")
print(output_dir.resolve())

Period 1 日期范围： 2025-06-03 06:02:57 到 2025-07-01 07:17:16
Period 2 日期范围： 2026-02-05 00:00:00 到 2026-04-23 18:02:00
计划下载股价范围： 2025-05-19 到 2026-06-22

9988.HK 股价数据：


Price,price_date,adj_close,close,high,low,open,volume,ticker
0,2025-05-19,118.949150,119.199997,120.800003,117.500000,119.900002,124103481,9988.HK
1,2025-05-20,121.443893,121.699997,122.599998,119.199997,120.000000,63599789,9988.HK
2,2025-05-21,122.840950,123.099998,123.699997,122.099998,123.099998,48764348,9988.HK
3,2025-05-22,118.849365,119.099998,121.800003,118.599998,121.599998,77869662,9988.HK
4,2025-05-23,118.550003,118.800003,119.900002,117.699997,119.500000,66708012,9988.HK


Price,price_date,adj_close,close,high,low,open,volume,ticker
243,2026-05-13,132.800003,132.800003,134.199997,130.300003,132.000000,89332520,9988.HK
244,2026-05-14,137.899994,137.899994,144.000000,137.399994,143.100006,171292701,9988.HK
245,2026-05-15,132.300003,132.300003,138.000000,131.100006,138.000000,90144879,9988.HK
246,2026-05-18,131.699997,131.699997,134.899994,128.600006,130.399994,94625028,9988.HK
247,2026-05-19,134.199997,134.199997,135.300003,130.500000,130.500000,33939456,9988.HK



HSTECH.HK 指数数据：


Price,price_date,adj_close,close,high,low,open,volume,ticker
0,2025-05-19,5.225,5.225,5.255,5.135,5.200,12004494,3032.HK
1,2025-05-20,5.290,5.290,5.305,5.215,5.225,9766702,3032.HK
2,2025-05-21,5.305,5.305,5.345,5.290,5.290,10845900,3032.HK
3,2025-05-22,5.215,5.215,5.310,5.195,5.300,13767398,3032.HK
4,2025-05-23,5.210,5.210,5.275,5.185,5.260,12209224,3032.HK


...


========== Period 1 with returns preview ==========


,article_id,date,event_date,sentiment_score,source_period,relevance_flag,stock_event_adj_close,ret_1d,ret_5d,ret_21d,hstech_ret_1d,hstech_ret_5d,hstech_ret_21d,abret_1d,abret_5d,abret_21d
0,period_1_2025_0200,2025-06-03 06:02:57,2025-06-03,3,period_1_2025,0,113.660309,0.006146,0.04302,-0.065637,0.005814,0.039729,0.00969,0.000332,0.003292,-0.075327
1,period_1_2025_0199,2025-06-03 08:01:43,2025-06-03,3,period_1_2025,0,113.660309,0.006146,0.04302,-0.065637,0.005814,0.039729,0.00969,0.000332,0.003292,-0.075327
2,period_1_2025_0198,2025-06-03 08:02:03,2025-06-03,3,period_1_2025,0,113.660309,0.006146,0.04302,-0.065637,0.005814,0.039729,0.00969,0.000332,0.003292,-0.075327
3,period_1_2025_0197,2025-06-03 08:23:36,2025-06-03,3,period_1_2025,0,113.660309,0.006146,0.04302,-0.065637,0.005814,0.039729,0.00969,0.000332,0.003292,-0.075327
4,period_1_2025_0196,2025-06-03 14:13:17,2025-06-03,4,period_1_2025,1,113.660309,0.006146,0.04302,-0.065637,0.005814,0.039729,0.00969,0.000332,0.003292,-0.075327



========== Period 2 with returns preview ==========


,article_id,date,event_date,sentiment_score,source_period,relevance_flag,stock_event_adj_close,ret_1d,ret_5d,ret_21d,hstech_ret_1d,hstech_ret_5d,hstech_ret_21d,abret_1d,abret_5d,abret_21d
0,period_2_2026_0001,2026-02-05 00:00:00,2026-02-05,2,period_2_2026,1,159.600006,-0.028822,-0.006266,-0.165414,-0.009311,0.003724,-0.065177,-0.019511,-0.009990,-0.100237
1,period_2_2026_0002,2026-02-05 00:00:00,2026-02-05,3,period_2_2026,1,159.600006,-0.028822,-0.006266,-0.165414,-0.009311,0.003724,-0.065177,-0.019511,-0.009990,-0.100237
2,period_2_2026_0003,2026-02-05 14:33:00,2026-02-05,2,period_2_2026,1,159.600006,-0.028822,-0.006266,-0.165414,-0.009311,0.003724,-0.065177,-0.019511,-0.009990,-0.100237
3,period_2_2026_0004,2026-02-05 22:14:00,2026-02-05,5,period_2_2026,1,159.600006,-0.028822,-0.006266,-0.165414,-0.009311,0.003724,-0.065177,-0.019511,-0.009990,-0.100237
4,period_2_2026_0008,2026-02-12 00:00:00,2026-02-12,3,period_2_2026,0,158.600006,-0.020177,-0.066835,-0.131778,-0.010204,-0.027829,-0.056586,-0.009973,-0.039006,-0.075192



Step 2 + Step 3 完成，文件已保存到：
/Users/sunny/Desktop/Capstone/中期汇报/step2_step3_price_return_outputs


In [8]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path

# =========================
# 1. 设置输入输出路径
# =========================

input_dir = Path("step1_cleaned_outputs")

period1_file = input_dir / "period_1_2025_cleaned.csv"
period2_file = input_dir / "period_2_2026_cleaned.csv"

output_dir = Path("step2_step3_price_return_outputs")
output_dir.mkdir(exist_ok=True)

# 主股票和市场基准
stock_ticker = "9988.HK"

# 这里使用 3032.HK 作为 Hang Seng TECH Index 的 ETF proxy
# 原来的 HSTECH.HK 在 yfinance 中历史数据不完整，所以不用
market_ticker = "3032.HK"
market_name = "3032_HK_HSTECH_ETF_PROXY"

# 需要计算的未来收益窗口
horizons = [1, 5, 21]


# =========================
# 2. 读取 Step 1 清洗后的新闻数据
# =========================

period1 = pd.read_csv(period1_file)
period2 = pd.read_csv(period2_file)

period1["date"] = pd.to_datetime(period1["date"], errors="coerce")
period2["date"] = pd.to_datetime(period2["date"], errors="coerce")

print("Period 1 日期范围：", period1["date"].min(), "到", period1["date"].max())
print("Period 2 日期范围：", period2["date"].min(), "到", period2["date"].max())


# =========================
# 3. 确定股价下载时间范围
# =========================

all_news_dates = pd.concat([period1["date"], period2["date"]]).dropna()

start_date = all_news_dates.min() - pd.Timedelta(days=15)

# 为了计算未来 21 个交易日收益，结束日期需要往后延长
# 如果当前日期还不够远，yfinance 会自动只下载到最新可获得交易日
end_date = all_news_dates.max() + pd.Timedelta(days=60)

print("计划下载股价范围：", start_date.date(), "到", end_date.date())


# =========================
# 4. 下载股价数据
# =========================

def download_price_data(ticker, start, end, min_rows=30):
    """
    下载日频股价数据。
    auto_adjust=False 是为了保留 Adj Close。
    min_rows 用来避免只下载到 1 行但代码误判成功。
    """

    data = yf.download(
        ticker,
        start=start.strftime("%Y-%m-%d"),
        end=end.strftime("%Y-%m-%d"),
        auto_adjust=False,
        progress=False
    )

    if data.empty:
        raise ValueError(f"{ticker} 下载失败，返回为空。请检查 ticker 是否正确。")

    # 如果 yfinance 返回多层列名，这里压平
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)

    data = data.reset_index()

    # 统一列名
    data = data.rename(columns={
        "Date": "price_date",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Adj Close": "adj_close",
        "Volume": "volume"
    })

    data["price_date"] = pd.to_datetime(data["price_date"])
    data = data.sort_values("price_date").reset_index(drop=True)

    # 如果没有 adj_close，就用 close 代替
    if "adj_close" not in data.columns or data["adj_close"].isna().all():
        data["adj_close"] = data["close"]

    data["ticker"] = ticker

    # 行数检查
    if len(data) < min_rows:
        raise ValueError(
            f"{ticker} 下载到的数据太少，只有 {len(data)} 行。"
            f"日期范围是 {data['price_date'].min()} 到 {data['price_date'].max()}。"
            "这通常说明该 ticker 不适合用 yfinance 下载历史数据。"
        )

    return data


stock_price = download_price_data(stock_ticker, start_date, end_date, min_rows=30)
market_price = download_price_data(market_ticker, start_date, end_date, min_rows=30)

print("\n9988.HK 股价数据：")
display(stock_price.head())
display(stock_price.tail())

print("\n3032.HK 市场基准数据：")
display(market_price.head())
display(market_price.tail())


# =========================
# 5. 检查下载结果
# =========================

print("9988.HK 数据日期范围：", stock_price["price_date"].min(), "到", stock_price["price_date"].max())
print("3032.HK 数据日期范围：", market_price["price_date"].min(), "到", market_price["price_date"].max())

print("9988.HK 行数：", len(stock_price))
print("3032.HK 行数：", len(market_price))


# =========================
# 6. 计算某个价格表的 forward return
# =========================

def add_forward_returns(price_df, horizons=[1, 5, 21], prefix="ret"):
    """
    对价格数据计算未来收益。
    ret_h = adj_close[t+h] / adj_close[t] - 1
    """

    df = price_df.copy()
    df = df.sort_values("price_date").reset_index(drop=True)

    for h in horizons:
        df[f"{prefix}_{h}d"] = df["adj_close"].shift(-h) / df["adj_close"] - 1

    return df


stock_price_ret = add_forward_returns(
    stock_price,
    horizons=horizons,
    prefix="stock_ret"
)

market_price_ret = add_forward_returns(
    market_price,
    horizons=horizons,
    prefix="market_ret"
)


# =========================
# 7. 给新闻匹配 event_date 和 forward return
# =========================

def attach_returns_to_news(news_df, stock_ret_df, market_ret_df, horizons=[1, 5, 21]):
    """
    对每条新闻：
    1. 将新闻 date 映射到下一个可交易日 event_date
    2. 匹配 9988.HK 的未来收益
    3. 匹配 3032.HK 市场基准的未来收益
    4. 计算 abnormal return

    abnormal return = Alibaba return - market benchmark return
    """

    news = news_df.copy()
    news["date"] = pd.to_datetime(news["date"], errors="coerce")
    news["news_date_only"] = news["date"].dt.normalize()

    stock = stock_ret_df.copy()
    market = market_ret_df.copy()

    stock["price_date"] = pd.to_datetime(stock["price_date"])
    market["price_date"] = pd.to_datetime(market["price_date"])

    stock_dates = stock["price_date"].sort_values().reset_index(drop=True)

    def get_next_trading_day(news_date):
        """
        如果新闻日期是交易日，则用当天；
        如果新闻日期不是交易日，则用下一个交易日。
        """

        if pd.isna(news_date):
            return pd.NaT

        possible_dates = stock_dates[stock_dates >= news_date]

        if len(possible_dates) == 0:
            return pd.NaT

        return possible_dates.iloc[0]

    news["event_date"] = news["news_date_only"].apply(get_next_trading_day)

    # 只保留需要合并的股价收益列
    stock_cols = ["price_date", "adj_close"] + [f"stock_ret_{h}d" for h in horizons]
    market_cols = ["price_date", "adj_close"] + [f"market_ret_{h}d" for h in horizons]

    stock_merge = stock[stock_cols].rename(columns={
        "price_date": "event_date",
        "adj_close": "stock_event_adj_close"
    })

    market_merge = market[market_cols].rename(columns={
        "price_date": "event_date",
        "adj_close": "market_event_adj_close"
    })

    news = news.merge(stock_merge, on="event_date", how="left")
    news = news.merge(market_merge, on="event_date", how="left")

    # 生成最终收益列
    for h in horizons:
        news[f"ret_{h}d"] = news[f"stock_ret_{h}d"]
        news[f"market_ret_{h}d"] = news[f"market_ret_{h}d"]
        news[f"abret_{h}d"] = news[f"ret_{h}d"] - news[f"market_ret_{h}d"]

    # 删除中间 stock_ret 列
    drop_cols = []
    for h in horizons:
        drop_cols.append(f"stock_ret_{h}d")

    news = news.drop(columns=drop_cols)

    # 按日期排序
    news = news.sort_values(["event_date", "date"]).reset_index(drop=True)

    return news


period1_with_returns = attach_returns_to_news(
    period1,
    stock_price_ret,
    market_price_ret,
    horizons=horizons
)

period2_with_returns = attach_returns_to_news(
    period2,
    stock_price_ret,
    market_price_ret,
    horizons=horizons
)


# =========================
# 8. 生成收益匹配质量报告
# =========================

def return_quality_report(df, name, horizons=[1, 5, 21]):
    report = {
        "source_period": name,
        "rows": len(df),
        "missing_event_date": df["event_date"].isna().sum(),
        "missing_stock_event_adj_close": df["stock_event_adj_close"].isna().sum(),
        "missing_market_event_adj_close": df["market_event_adj_close"].isna().sum(),
    }

    for h in horizons:
        report[f"missing_ret_{h}d"] = df[f"ret_{h}d"].isna().sum()
        report[f"missing_market_ret_{h}d"] = df[f"market_ret_{h}d"].isna().sum()
        report[f"missing_abret_{h}d"] = df[f"abret_{h}d"].isna().sum()

    return report


quality_returns = pd.DataFrame([
    return_quality_report(period1_with_returns, "period_1_2025", horizons),
    return_quality_report(period2_with_returns, "period_2_2026", horizons)
])

print("\n========== 收益匹配质量报告 ==========")
display(quality_returns)


# =========================
# 9. 查看收益列是否正常
# =========================

return_cols = [
    "article_id",
    "date",
    "event_date",
    "sentiment_score",
    "source_period",
    "relevance_flag",
    "stock_event_adj_close",
    "market_event_adj_close",
    "ret_1d",
    "ret_5d",
    "ret_21d",
    "market_ret_1d",
    "market_ret_5d",
    "market_ret_21d",
    "abret_1d",
    "abret_5d",
    "abret_21d",
]

print("\n========== Period 1 with returns preview ==========")
display(period1_with_returns[return_cols].head())

print("\n========== Period 2 with returns preview ==========")
display(period2_with_returns[return_cols].head())


# =========================
# 10. 保存 Step 2 + Step 3 输出
# =========================

stock_price_ret.to_csv(
    output_dir / "9988_HK_price_with_forward_returns.csv",
    index=False,
    encoding="utf-8-sig"
)

market_price_ret.to_csv(
    output_dir / f"{market_name}_price_with_forward_returns.csv",
    index=False,
    encoding="utf-8-sig"
)

period1_with_returns.to_csv(
    output_dir / "period_1_2025_with_returns.csv",
    index=False,
    encoding="utf-8-sig"
)

period2_with_returns.to_csv(
    output_dir / "period_2_2026_with_returns.csv",
    index=False,
    encoding="utf-8-sig"
)

quality_returns.to_csv(
    output_dir / "step2_step3_return_quality_report.csv",
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(output_dir / "step2_step3_price_return_outputs.xlsx", engine="openpyxl") as writer:
    stock_price_ret.to_excel(writer, sheet_name="9988_HK_price", index=False)
    market_price_ret.to_excel(writer, sheet_name="3032_HK_market_proxy", index=False)
    period1_with_returns.to_excel(writer, sheet_name="period_1_with_returns", index=False)
    period2_with_returns.to_excel(writer, sheet_name="period_2_with_returns", index=False)
    quality_returns.to_excel(writer, sheet_name="quality_report", index=False)

print("\nStep 2 + Step 3 完成，文件已保存到：")
print(output_dir.resolve())

Period 1 日期范围： 2025-06-03 06:02:57 到 2025-07-01 07:17:16
Period 2 日期范围： 2026-02-05 00:00:00 到 2026-04-23 18:02:00
计划下载股价范围： 2025-05-19 到 2026-06-22

9988.HK 股价数据：


Price,price_date,adj_close,close,high,low,open,volume,ticker
0,2025-05-19,118.949150,119.199997,120.800003,117.500000,119.900002,124103481,9988.HK
1,2025-05-20,121.443893,121.699997,122.599998,119.199997,120.000000,63599789,9988.HK
2,2025-05-21,122.840950,123.099998,123.699997,122.099998,123.099998,48764348,9988.HK
3,2025-05-22,118.849365,119.099998,121.800003,118.599998,121.599998,77869662,9988.HK
4,2025-05-23,118.550003,118.800003,119.900002,117.699997,119.500000,66708012,9988.HK


Price,price_date,adj_close,close,high,low,open,volume,ticker
243,2026-05-13,132.800003,132.800003,134.199997,130.300003,132.000000,89332520,9988.HK
244,2026-05-14,137.899994,137.899994,144.000000,137.399994,143.100006,171292701,9988.HK
245,2026-05-15,132.300003,132.300003,138.000000,131.100006,138.000000,90144879,9988.HK
246,2026-05-18,131.699997,131.699997,134.899994,128.600006,130.399994,94625028,9988.HK
247,2026-05-19,134.699997,134.699997,135.300003,130.500000,130.500000,36396348,9988.HK



3032.HK 市场基准数据：


Price,price_date,adj_close,close,high,low,open,volume,ticker
0,2025-05-19,5.225,5.225,5.255,5.135,5.200,12004494,3032.HK
1,2025-05-20,5.290,5.290,5.305,5.215,5.225,9766702,3032.HK
2,2025-05-21,5.305,5.305,5.345,5.290,5.290,10845900,3032.HK
3,2025-05-22,5.215,5.215,5.310,5.195,5.300,13767398,3032.HK
4,2025-05-23,5.210,5.210,5.275,5.185,5.260,12209224,3032.HK


...


========== Period 1 with returns preview ==========


,article_id,date,event_date,sentiment_score,source_period,relevance_flag,stock_event_adj_close,market_event_adj_close,ret_1d,ret_5d,ret_21d,market_ret_1d,market_ret_5d,market_ret_21d,abret_1d,abret_5d,abret_21d
0,period_1_2025_0200,2025-06-03 06:02:57,2025-06-03,3,period_1_2025,0,113.660309,5.16,0.006146,0.04302,-0.065637,0.005814,0.039729,0.00969,0.000332,0.003292,-0.075327
1,period_1_2025_0199,2025-06-03 08:01:43,2025-06-03,3,period_1_2025,0,113.660309,5.16,0.006146,0.04302,-0.065637,0.005814,0.039729,0.00969,0.000332,0.003292,-0.075327
2,period_1_2025_0198,2025-06-03 08:02:03,2025-06-03,3,period_1_2025,0,113.660309,5.16,0.006146,0.04302,-0.065637,0.005814,0.039729,0.00969,0.000332,0.003292,-0.075327
3,period_1_2025_0197,2025-06-03 08:23:36,2025-06-03,3,period_1_2025,0,113.660309,5.16,0.006146,0.04302,-0.065637,0.005814,0.039729,0.00969,0.000332,0.003292,-0.075327
4,period_1_2025_0196,2025-06-03 14:13:17,2025-06-03,4,period_1_2025,1,113.660309,5.16,0.006146,0.04302,-0.065637,0.005814,0.039729,0.00969,0.000332,0.003292,-0.075327



========== Period 2 with returns preview ==========


,article_id,date,event_date,sentiment_score,source_period,relevance_flag,stock_event_adj_close,market_event_adj_close,ret_1d,ret_5d,ret_21d,market_ret_1d,market_ret_5d,market_ret_21d,abret_1d,abret_5d,abret_21d
0,period_2_2026_0001,2026-02-05 00:00:00,2026-02-05,2,period_2_2026,1,159.600006,5.37,-0.028822,-0.006266,-0.165414,-0.009311,0.003724,-0.065177,-0.019511,-0.009990,-0.100237
1,period_2_2026_0002,2026-02-05 00:00:00,2026-02-05,3,period_2_2026,1,159.600006,5.37,-0.028822,-0.006266,-0.165414,-0.009311,0.003724,-0.065177,-0.019511,-0.009990,-0.100237
2,period_2_2026_0003,2026-02-05 14:33:00,2026-02-05,2,period_2_2026,1,159.600006,5.37,-0.028822,-0.006266,-0.165414,-0.009311,0.003724,-0.065177,-0.019511,-0.009990,-0.100237
3,period_2_2026_0004,2026-02-05 22:14:00,2026-02-05,5,period_2_2026,1,159.600006,5.37,-0.028822,-0.006266,-0.165414,-0.009311,0.003724,-0.065177,-0.019511,-0.009990,-0.100237
4,period_2_2026_0008,2026-02-12 00:00:00,2026-02-12,3,period_2_2026,0,158.600006,5.39,-0.020177,-0.066835,-0.131778,-0.010204,-0.027829,-0.056586,-0.009973,-0.039006,-0.075192



Step 2 + Step 3 完成，文件已保存到：
/Users/sunny/Desktop/Capstone/中期汇报/step2_step3_price_return_outputs


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# Step 4：合并两个已经完成收益匹配的新闻数据集
# =========================

input_dir = Path("step2_step3_price_return_outputs")
output_dir = Path("step4_final_dataset_outputs")
output_dir.mkdir(exist_ok=True)

period1_file = input_dir / "period_1_2025_with_returns.csv"
period2_file = input_dir / "period_2_2026_with_returns.csv"

period1 = pd.read_csv(period1_file)
period2 = pd.read_csv(period2_file)

# 日期格式修正
for df in [period1, period2]:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")

print("Period 1 shape:", period1.shape)
print("Period 2 shape:", period2.shape)

print("\nPeriod 1 日期范围：", period1["date"].min(), "到", period1["date"].max())
print("Period 2 日期范围：", period2["date"].min(), "到", period2["date"].max())


# =========================
# 1. 指定最终需要保留的核心字段
# =========================

final_columns = [
    "article_id",
    "date",
    "event_date",
    "title_en",
    "body_en",
    "full_text",
    "sentiment_score",
    "sentiment_reason",
    "source_period",
    "relevance_flag",
    "ticker",
    "ret_1d",
    "ret_5d",
    "ret_21d",
    "abret_1d",
    "abret_5d",
    "abret_21d",
]

# 如果某些列不存在，自动补 NaN，避免报错
for df in [period1, period2]:
    for col in final_columns:
        if col not in df.columns:
            df[col] = np.nan

period1_core = period1[final_columns].copy()
period2_core = period2[final_columns].copy()


# =========================
# 2. 合并两个周期
# =========================

final_all = pd.concat([period1_core, period2_core], ignore_index=True)

# 按 event_date 和 date 排序
final_all = final_all.sort_values(["event_date", "date"], na_position="last").reset_index(drop=True)

print("\n合并后总样本量：", len(final_all))
display(final_all.head())


# =========================
# 3. 基础缺失值检查
# =========================

key_columns = [
    "date",
    "event_date",
    "title_en",
    "body_en",
    "full_text",
    "sentiment_score",
    "sentiment_reason",
    "source_period",
    "relevance_flag",
    "ret_1d",
    "ret_5d",
    "ret_21d",
    "abret_1d",
    "abret_5d",
    "abret_21d",
]

missing_summary = (
    final_all[key_columns]
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)

missing_summary["missing_pct"] = missing_summary["missing_count"] / len(final_all)

print("\n========== 合并后缺失值检查 ==========")
display(missing_summary)


# =========================
# 4. 检查 date / event_date 缺失样本
# =========================

missing_date_rows = final_all[
    final_all["date"].isna() | final_all["event_date"].isna()
].copy()

print("\n缺失 date 或 event_date 的样本数量：", len(missing_date_rows))

if len(missing_date_rows) > 0:
    display(
        missing_date_rows[
            [
                "article_id",
                "date",
                "event_date",
                "title_en",
                "sentiment_score",
                "source_period",
                "relevance_flag",
            ]
        ]
    )


# =========================
# 5. 检查 sentiment_score 分布是否一致
# =========================

score_distribution_count = (
    final_all
    .groupby(["source_period", "sentiment_score"])
    .size()
    .reset_index(name="count")
)

score_distribution_pct = (
    score_distribution_count
    .copy()
)

score_distribution_pct["pct_within_period"] = (
    score_distribution_pct["count"]
    / score_distribution_pct.groupby("source_period")["count"].transform("sum")
)

print("\n========== sentiment_score 数量分布 ==========")
display(score_distribution_count)

print("\n========== sentiment_score 占比分布 ==========")
display(score_distribution_pct)


# 透视表版本，更直观
score_pivot_count = pd.crosstab(
    final_all["source_period"],
    final_all["sentiment_score"],
    margins=True
)

score_pivot_pct = pd.crosstab(
    final_all["source_period"],
    final_all["sentiment_score"],
    normalize="index"
)

print("\n========== sentiment_score 透视表：数量 ==========")
display(score_pivot_count)

print("\n========== sentiment_score 透视表：行内占比 ==========")
display(score_pivot_pct)


# =========================
# 6. 检查 relevance_flag 分布
# =========================

relevance_distribution = pd.crosstab(
    final_all["source_period"],
    final_all["relevance_flag"],
    margins=True
)

relevance_distribution_pct = pd.crosstab(
    final_all["source_period"],
    final_all["relevance_flag"],
    normalize="index"
)

print("\n========== relevance_flag 数量分布 ==========")
display(relevance_distribution)

print("\n========== relevance_flag 行内占比 ==========")
display(relevance_distribution_pct)


# =========================
# 7. 检查收益分布是否存在明显差异
# =========================

return_cols = [
    "ret_1d",
    "ret_5d",
    "ret_21d",
    "abret_1d",
    "abret_5d",
    "abret_21d",
]

return_distribution = (
    final_all
    .groupby("source_period")[return_cols]
    .agg(["count", "mean", "std", "min", "median", "max"])
)

print("\n========== 两个周期的收益分布对比 ==========")
display(return_distribution)


# 更简洁的均值对比
return_mean_compare = (
    final_all
    .groupby("source_period")[return_cols]
    .mean()
    .reset_index()
)

print("\n========== 两个周期的平均收益对比 ==========")
display(return_mean_compare)


# =========================
# 8. 按情绪组检查平均收益
# =========================

# 三分类情绪标签：更适合后续分析
def to_sentiment_3class(score):
    if pd.isna(score):
        return np.nan
    if score in [1, 2]:
        return "negative"
    elif score == 3:
        return "neutral"
    elif score in [4, 5]:
        return "positive"
    else:
        return np.nan

final_all["sentiment_3class"] = final_all["sentiment_score"].apply(to_sentiment_3class)

sentiment_return_summary = (
    final_all
    .groupby(["source_period", "sentiment_3class"])[return_cols]
    .agg(["count", "mean", "median"])
)

print("\n========== 按三分类情绪分组的收益表现 ==========")
display(sentiment_return_summary)


# =========================
# 9. 简单 dataset shift 检查
# =========================

dataset_shift_summary = pd.DataFrame({
    "metric": [
        "sample_size",
        "date_min",
        "date_max",
        "mean_sentiment_score",
        "pct_relevance_flag_1",
        "mean_ret_1d",
        "mean_ret_5d",
        "mean_ret_21d",
        "mean_abret_1d",
        "mean_abret_5d",
        "mean_abret_21d",
    ]
})

shift_rows = []

for period_name, group in final_all.groupby("source_period"):
    row = {
        "source_period": period_name,
        "sample_size": len(group),
        "date_min": group["date"].min(),
        "date_max": group["date"].max(),
        "mean_sentiment_score": group["sentiment_score"].mean(),
        "pct_relevance_flag_1": group["relevance_flag"].mean(),
        "mean_ret_1d": group["ret_1d"].mean(),
        "mean_ret_5d": group["ret_5d"].mean(),
        "mean_ret_21d": group["ret_21d"].mean(),
        "mean_abret_1d": group["abret_1d"].mean(),
        "mean_abret_5d": group["abret_5d"].mean(),
        "mean_abret_21d": group["abret_21d"].mean(),
    }
    shift_rows.append(row)

dataset_shift_summary = pd.DataFrame(shift_rows)

print("\n========== Dataset Shift 初步检查 ==========")
display(dataset_shift_summary)


# =========================
# 10. 构造 return-ready 数据集
# =========================

# 这个版本用于后续收益分析和端到端回报预测
# 要求 date / event_date / ret_1d / ret_5d / abret_1d / abret_5d 不缺失
# ret_21d 和 abret_21d 不作为强制条件，因为 Period 2 后段可能自然缺失

required_for_return_modeling = [
    "date",
    "event_date",
    "full_text",
    "sentiment_score",
    "ret_1d",
    "ret_5d",
    "abret_1d",
    "abret_5d",
]

final_return_ready = final_all.dropna(subset=required_for_return_modeling).copy()
final_return_ready = final_return_ready.sort_values(["event_date", "date"]).reset_index(drop=True)

print("\n========== Return-ready 数据集 ==========")
print("final_all 样本量：", len(final_all))
print("final_return_ready 样本量：", len(final_return_ready))
print("删除样本数：", len(final_all) - len(final_return_ready))

display(final_return_ready.head())


# =========================
# 11. 相关样本版本：relevance_flag = 1
# =========================

final_all_relevant = final_all[final_all["relevance_flag"] == 1].copy()
final_return_ready_relevant = final_return_ready[final_return_ready["relevance_flag"] == 1].copy()

print("\n========== relevance_flag = 1 样本量 ==========")
print("final_all_relevant 样本量：", len(final_all_relevant))
print("final_return_ready_relevant 样本量：", len(final_return_ready_relevant))


# =========================
# 12. 保存最终数据集
# =========================

# 检查表不保存，只保存后续建模需要的数据集
final_all.to_csv(
    output_dir / "final_alibaba_news_dataset_all.csv",
    index=False,
    encoding="utf-8-sig"
)

final_return_ready.to_csv(
    output_dir / "final_alibaba_news_dataset_return_ready.csv",
    index=False,
    encoding="utf-8-sig"
)

final_all_relevant.to_csv(
    output_dir / "final_alibaba_news_dataset_all_relevant_only.csv",
    index=False,
    encoding="utf-8-sig"
)

final_return_ready_relevant.to_csv(
    output_dir / "final_alibaba_news_dataset_return_ready_relevant_only.csv",
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(output_dir / "step4_final_alibaba_news_datasets.xlsx", engine="openpyxl") as writer:
    final_all.to_excel(writer, sheet_name="all_samples", index=False)
    final_return_ready.to_excel(writer, sheet_name="return_ready", index=False)
    final_all_relevant.to_excel(writer, sheet_name="all_relevant_only", index=False)
    final_return_ready_relevant.to_excel(writer, sheet_name="return_ready_relevant", index=False)

print("\nStep 4 完成，最终数据集已保存到：")
print(output_dir.resolve())

Period 1 shape: (200, 25)
Period 2 shape: (176, 25)

Period 1 日期范围： 2025-06-03 06:02:57 到 2025-07-01 07:17:16
Period 2 日期范围： 2026-02-05 00:00:00 到 2026-04-23 18:02:00

合并后总样本量： 376


,article_id,date,event_date,title_en,body_en,full_text,sentiment_score,sentiment_reason,source_period,relevance_flag,ticker,ret_1d,ret_5d,ret_21d,abret_1d,abret_5d,abret_21d
0,period_1_2025_0200,2025-06-03 06:02:57,2025-06-03,Platform economy stimulates the vitality of sm...,Individual industrial and commercial household...,Platform economy stimulates the vitality of sm...,3,Unrelated to Alibaba company entities,period_1_2025,0,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327
1,period_1_2025_0199,2025-06-03 08:01:43,2025-06-03,"""Underage movie viewing discount policy 'hide ...","Behind a regular movie ticket, there is a hidd...","""Underage movie viewing discount policy 'hide ...",3,Unrelated to Alibaba company entities,period_1_2025,0,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327
2,period_1_2025_0198,2025-06-03 08:02:03,2025-06-03,Setting up a scam in the name of borrowing a h...,"Self-proclaimed as the ""red envelope snatching...",Setting up a scam in the name of borrowing a h...,3,Unrelated to Alibaba company entities,period_1_2025,0,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327
3,period_1_2025_0197,2025-06-03 08:23:36,2025-06-03,Notice on holding the 2024 Academic Annual Con...,Relevant units: Pharmaceutical excipients are ...,Notice on holding the 2024 Academic Annual Con...,3,Unrelated to Alibaba company entities,period_1_2025,0,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327
4,period_1_2025_0196,2025-06-03 14:13:17,2025-06-03,"Seize the 90-day window period ""Foreign Trade ...",China Business News reporter Li Li reported fr...,"Seize the 90-day window period ""Foreign Trade ...",4,"Taobao's 618 event is being promoted globally,...",period_1_2025,1,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327



========== 合并后缺失值检查 ==========


,column,missing_count,missing_pct
0,date,6,0.015957
1,event_date,6,0.015957
2,title_en,0,0.000000
3,body_en,0,0.000000
4,full_text,0,0.000000
5,sentiment_score,0,0.000000
6,sentiment_reason,0,0.000000
7,source_period,0,0.000000
8,relevance_flag,0,0.000000
9,ret_1d,6,0.015957



缺失 date 或 event_date 的样本数量： 6


...


========== Dataset Shift 初步检查 ==========


,source_period,sample_size,date_min,date_max,mean_sentiment_score,pct_relevance_flag_1,mean_ret_1d,mean_ret_5d,mean_ret_21d,mean_abret_1d,mean_abret_5d,mean_abret_21d
0,period_1_2025,200,2025-06-03 06:02:57,2025-07-01 07:17:16,3.170000,0.205000,-0.001113,-0.022567,-0.006088,-0.002338,-0.024240,-0.041797
1,period_2_2026,176,2026-02-05 00:00:00,2026-04-23 18:02:00,3.039773,0.642045,-0.005882,-0.024041,-0.023939,-0.000438,-0.008972,-0.008295



========== Return-ready 数据集 ==========
final_all 样本量： 376
final_return_ready 样本量： 370
删除样本数： 6


,article_id,date,event_date,title_en,body_en,full_text,sentiment_score,sentiment_reason,source_period,relevance_flag,ticker,ret_1d,ret_5d,ret_21d,abret_1d,abret_5d,abret_21d,sentiment_3class
0,period_1_2025_0200,2025-06-03 06:02:57,2025-06-03,Platform economy stimulates the vitality of sm...,Individual industrial and commercial household...,Platform economy stimulates the vitality of sm...,3,Unrelated to Alibaba company entities,period_1_2025,0,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327,neutral
1,period_1_2025_0199,2025-06-03 08:01:43,2025-06-03,"""Underage movie viewing discount policy 'hide ...","Behind a regular movie ticket, there is a hidd...","""Underage movie viewing discount policy 'hide ...",3,Unrelated to Alibaba company entities,period_1_2025,0,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327,neutral
2,period_1_2025_0198,2025-06-03 08:02:03,2025-06-03,Setting up a scam in the name of borrowing a h...,"Self-proclaimed as the ""red envelope snatching...",Setting up a scam in the name of borrowing a h...,3,Unrelated to Alibaba company entities,period_1_2025,0,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327,neutral
3,period_1_2025_0197,2025-06-03 08:23:36,2025-06-03,Notice on holding the 2024 Academic Annual Con...,Relevant units: Pharmaceutical excipients are ...,Notice on holding the 2024 Academic Annual Con...,3,Unrelated to Alibaba company entities,period_1_2025,0,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327,neutral
4,period_1_2025_0196,2025-06-03 14:13:17,2025-06-03,"Seize the 90-day window period ""Foreign Trade ...",China Business News reporter Li Li reported fr...,"Seize the 90-day window period ""Foreign Trade ...",4,"Taobao's 618 event is being promoted globally,...",period_1_2025,1,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327,positive



========== relevance_flag = 1 样本量 ==========
final_all_relevant 样本量： 154
final_return_ready_relevant 样本量： 150

Step 4 完成，最终数据集已保存到：
/Users/sunny/Desktop/Capstone/中期汇报/step4_final_dataset_outputs


---


# **Sentiment Label**

### **BERT Sentiment Classifier**

In [10]:
# 如果你还没有安装这些包，先运行这一格
!pip install transformers datasets accelerate scikit-learn openpyxl -q

In [19]:
# =========================
# Step 5：BERT Sentiment Classifier
# 简洁输出版
# =========================

!pip install transformers datasets accelerate scikit-learn openpyxl -q

import pandas as pd
import numpy as np
from pathlib import Path
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, mean_absolute_error

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# =========================
# 1. 读取数据
# =========================

input_file = Path("step4_final_dataset_outputs/final_alibaba_news_dataset_all.csv")
output_dir = Path("step5_bert_sentiment_outputs_clean")
output_dir.mkdir(exist_ok=True)

df = pd.read_csv(input_file)

print("原始样本量:", len(df))
print("GPU 是否可用:", torch.cuda.is_available())

# =========================
# 2. 清洗并构造三分类标签
# 2 = negative, 3 = neutral, 4 = positive
# =========================

df = df.dropna(subset=["full_text", "sentiment_score"]).copy()
df["full_text"] = df["full_text"].astype(str).str.strip()
df = df[df["full_text"] != ""].copy()

df["sentiment_score"] = pd.to_numeric(df["sentiment_score"], errors="coerce")
df = df.dropna(subset=["sentiment_score"]).copy()
df["sentiment_score"] = df["sentiment_score"].astype(int)
df = df[df["sentiment_score"].between(1, 5)].copy()

def to_sentiment_3class(score):
    if score in [1, 2]:
        return 2
    elif score == 3:
        return 3
    elif score in [4, 5]:
        return 4
    else:
        return np.nan

df["sentiment_3class"] = df["sentiment_score"].apply(to_sentiment_3class)

print("\n五分类原始标签分布:")
display(df["sentiment_score"].value_counts().sort_index())

print("\n三分类标签分布:")
display(df["sentiment_3class"].value_counts().sort_index())

# =========================
# 3. 模型设置
# =========================

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label2id = {2: 0, 3: 1, 4: 2}
id2label = {0: 2, 1: 3, 2: 4}

df["label"] = df["sentiment_3class"].map(label2id)

# =========================
# 4. 划分 train / validation / test = 6:2:2
# =========================

train_val, test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment_3class"]
)

train, val = train_test_split(
    train_val,
    test_size=0.25,
    random_state=42,
    stratify=train_val["sentiment_3class"]
)

print("\n数据集划分:")
print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

# =========================
# 5. 转换为 HuggingFace Dataset
# =========================

keep_cols = ["full_text", "label"]

train_ds = Dataset.from_pandas(train[keep_cols])
val_ds = Dataset.from_pandas(val[keep_cols])
test_ds = Dataset.from_pandas(test[keep_cols])

def tokenize_function(examples):
    return tokenizer(
        examples["full_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================
# 6. 建立模型
# =========================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True
)

# =========================
# 7. 指标函数
# =========================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0)
    }

# =========================
# 8. 训练
# =========================

training_args = TrainingArguments(
    output_dir=str(output_dir / "training_checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# =========================
# 9. 测试集预测
# =========================

pred_output = trainer.predict(test_ds)

logits = pred_output.predictions
true_ids = pred_output.label_ids
pred_ids = np.argmax(logits, axis=1)

true_labels = [id2label[x] for x in true_ids]
pred_labels = [id2label[x] for x in pred_ids]

probs = torch.nn.functional.softmax(
    torch.tensor(logits),
    dim=-1
).numpy()

# =========================
# 10. 生成核心结果表
# =========================

accuracy = accuracy_score(true_ids, pred_ids)
macro_f1 = f1_score(true_ids, pred_ids, average="macro", zero_division=0)
weighted_f1 = f1_score(true_ids, pred_ids, average="weighted", zero_division=0)
mae = mean_absolute_error(true_labels, pred_labels)

metrics_summary = pd.DataFrame([{
    "model": MODEL_NAME,
    "task": "3-class sentiment classification",
    "label_definition": "1/2→2 negative, 3→3 neutral, 4/5→4 positive",
    "train_size": len(train),
    "validation_size": len(val),
    "test_size": len(test),
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "mae": mae
}])

cm = confusion_matrix(true_labels, pred_labels, labels=[2, 3, 4])

confusion_df = pd.DataFrame(
    cm,
    index=["true_2_negative", "true_3_neutral", "true_4_positive"],
    columns=["pred_2_negative", "pred_3_neutral", "pred_4_positive"]
)

test_predictions = test.copy().reset_index(drop=True)

test_predictions["true_sentiment_3class"] = true_labels
test_predictions["pred_sentiment_3class"] = pred_labels
test_predictions["correct_flag"] = (
    test_predictions["true_sentiment_3class"]
    == test_predictions["pred_sentiment_3class"]
).astype(int)

test_predictions["prob_2_negative"] = probs[:, 0]
test_predictions["prob_3_neutral"] = probs[:, 1]
test_predictions["prob_4_positive"] = probs[:, 2]

# 保留清晰字段，删掉内部 label
prediction_columns = [
    "article_id",
    "date",
    "title_en",
    "body_en",
    "full_text",
    "sentiment_score",
    "sentiment_3class",
    "true_sentiment_3class",
    "pred_sentiment_3class",
    "correct_flag",
    "prob_2_negative",
    "prob_3_neutral",
    "prob_4_positive",
    "source_period",
    "relevance_flag",
]

prediction_columns = [
    col for col in prediction_columns
    if col in test_predictions.columns
]

test_predictions_clean = test_predictions[prediction_columns].copy()

# =========================
# 11. Notebook 内显示结果
# =========================

print("\n========== Metrics Summary ==========")
display(metrics_summary)

print("\n========== Confusion Matrix ==========")
display(confusion_df)

print("\n========== Classification Report ==========")
print(classification_report(true_labels, pred_labels, labels=[2, 3, 4], zero_division=0))

print("\n========== Test Predictions Preview ==========")
display(test_predictions_clean.head())

# =========================
# 12. 保存核心产出
# =========================

# 1. 保存指标
metrics_summary.to_csv(
    output_dir / "bert_3class_metrics_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. 保存混淆矩阵
confusion_df.to_csv(
    output_dir / "bert_3class_confusion_matrix.csv",
    encoding="utf-8-sig"
)

# 3. 保存测试集预测
test_predictions_clean.to_csv(
    output_dir / "bert_3class_test_predictions_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

# 4. 保存 Excel 汇总
with pd.ExcelWriter(output_dir / "step5_bert_sentiment_outputs_clean.xlsx", engine="openpyxl") as writer:
    metrics_summary.to_excel(writer, sheet_name="metrics_summary", index=False)
    confusion_df.to_excel(writer, sheet_name="confusion_matrix")
    test_predictions_clean.to_excel(writer, sheet_name="test_predictions", index=False)

# 5. 保存模型，后续如不需要部署可以忽略这个文件夹
model_dir = output_dir / "best_distilbert_3class_model"
trainer.save_model(str(model_dir))
tokenizer.save_pretrained(str(model_dir))

# 6. 保存解释文件
readme_text = """
Step 5 Output Explanation
=========================

This folder contains the cleaned outputs of the BERT-based sentiment classification task.

Task Definition
---------------
Input:
- full_text = news title + news body

Target label:
- sentiment_3class

Label mapping:
- Original sentiment_score 1 or 2 → 2 = negative
- Original sentiment_score 3      → 3 = neutral
- Original sentiment_score 4 or 5 → 4 = positive

The model learns to reproduce the human-labeled golden standard sentiment labels.

Files
-----

1. bert_3class_metrics_summary.csv
   This file summarizes the overall test performance of the BERT sentiment classifier.

   Columns:
   - model: pretrained model used for fine-tuning
   - task: classification task name
   - label_definition: mapping from original 1–5 sentiment scores to 3-class labels
   - train_size: number of training samples
   - validation_size: number of validation samples
   - test_size: number of test samples
   - accuracy: proportion of correctly predicted test samples
   - macro_f1: unweighted average F1 across the three classes; important for imbalanced labels
   - weighted_f1: F1 score weighted by class sample size
   - mae: mean absolute error between true and predicted numeric labels

2. bert_3class_confusion_matrix.csv
   This file shows the structure of classification errors.

   Rows are true labels.
   Columns are predicted labels.

   Example:
   - true_4_positive / pred_3_neutral means that a truly positive news article was predicted as neutral.

3. bert_3class_test_predictions_clean.csv
   This file contains article-level predictions on the held-out test set.

   Columns:
   - article_id: unique article identifier
   - date: original news date
   - title_en: English news title
   - body_en: English news body
   - full_text: model input text
   - sentiment_score: original human label from 1 to 5
   - sentiment_3class: converted human label using 2/3/4 classes
   - true_sentiment_3class: true label in the test set
   - pred_sentiment_3class: BERT-predicted label
   - correct_flag: 1 if prediction is correct, 0 otherwise
   - prob_2_negative: predicted probability of class 2 negative
   - prob_3_neutral: predicted probability of class 3 neutral
   - prob_4_positive: predicted probability of class 4 positive
   - source_period: whether the article is from period 1 or period 2
   - relevance_flag: whether the article is judged as Alibaba-relevant by rule-based screening

4. step5_bert_sentiment_outputs_clean.xlsx
   Excel version combining the three key outputs:
   - metrics_summary
   - confusion_matrix
   - test_predictions

5. best_distilbert_3class_model/
   Saved fine-tuned DistilBERT model and tokenizer.
   This is useful only if the trained model needs to be reused for future prediction.

How to interpret the results
----------------------------
- Accuracy measures overall correctness.
- Macro F1 is especially important because the dataset is imbalanced.
- Weighted F1 reflects overall performance while giving more weight to larger classes.
- MAE measures how far the predicted sentiment label is from the human label numerically.
- The confusion matrix shows whether the model tends to over-predict neutral or confuse positive and negative news.
- The probability columns can later be used in return analysis, for example testing whether prob_4_positive is associated with higher future abnormal returns.
"""

with open(output_dir / "README_step5_outputs.txt", "w", encoding="utf-8") as f:
    f.write(readme_text)

print("\nStep 5 完成。核心输出已保存到：")
print(output_dir.resolve())

print("\n保存文件：")
print("1. bert_3class_metrics_summary.csv")
print("2. bert_3class_confusion_matrix.csv")
print("3. bert_3class_test_predictions_clean.csv")
print("4. step5_bert_sentiment_outputs_clean.xlsx")
print("5. README_step5_outputs.txt")
print("6. best_distilbert_3class_model/")

原始样本量: 376
GPU 是否可用: False

五分类原始标签分布:


sentiment_score
1      2
2     51
3    236
4     78
5      9
Name: count, dtype: int64


三分类标签分布:


sentiment_3class
2     53
3    236
4     87
Name: count, dtype: int64


数据集划分:
Train: 225
Validation: 75
Test: 76


...

,pred_2_negative,pred_3_neutral,pred_4_positive
true_2_negative,3,5,3
true_3_neutral,0,44,4
true_4_positive,0,6,11



========== Classification Report ==========
              precision    recall  f1-score   support

           2       1.00      0.27      0.43        11
           3       0.80      0.92      0.85        48
           4       0.61      0.65      0.63        17

    accuracy                           0.76        76
   macro avg       0.80      0.61      0.64        76
weighted avg       0.79      0.76      0.74        76


========== Test Predictions Preview ==========


,article_id,date,title_en,body_en,full_text,sentiment_score,sentiment_3class,true_sentiment_3class,pred_sentiment_3class,correct_flag,prob_2_negative,prob_3_neutral,prob_4_positive,source_period,relevance_flag
0,period_1_2025_0087,2025-06-19 03:01:29,"Some regions cancel ""national subsidy""? There ...","During the ""618"" period, online and offline pr...","Some regions cancel ""national subsidy""? There ...",3,3,3,3,1,0.096606,0.817779,0.085616,period_1_2025,0
1,period_1_2025_0016,2025-06-27 14:18:18,"Involved in 126 million yuan case, hundreds of...","With both hands leaning on a shiny cane, Wu Xi...","Involved in 126 million yuan case, hundreds of...",3,3,3,3,1,0.109842,0.817414,0.072744,period_1_2025,0
2,period_2_2026_0143,2026-04-16 07:00:00,"21 Interview | Lu Chuncong, President of the C...","Since the beginning of this year, the intellig...","21 Interview | Lu Chuncong, President of the C...",4,4,4,3,0,0.113806,0.725117,0.161077,period_2_2026,0
3,period_1_2025_0157,2025-06-07 06:03:29,"The Shanghai Cooperation Organization Youth ""P...","From June 2nd to 6th, the Shanghai Cooperation...","The Shanghai Cooperation Organization Youth ""P...",3,3,3,3,1,0.093953,0.848838,0.057209,period_1_2025,0
4,period_2_2026_0140,2026-04-15 00:00:00,Exploration of the Difficulties and Solutions ...,The deep integration of digital technology int...,Exploration of the Difficulties and Solutions ...,2,2,2,3,0,0.101279,0.732969,0.165752,period_2_2026,0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Step 5 完成。核心输出已保存到：
/Users/sunny/Desktop/Capstone/中期汇报/step5_bert_sentiment_outputs_clean

保存文件：
1. bert_3class_metrics_summary.csv
2. bert_3class_confusion_matrix.csv
3. bert_3class_test_predictions_clean.csv
4. step5_bert_sentiment_outputs_clean.xlsx
5. README_step5_outputs.txt
6. best_distilbert_3class_model/


In [20]:
import pandas as pd
import numpy as np
from pathlib import Path

from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import ttest_ind

import statsmodels.api as sm

# =========================
# Step7
# Sentiment → Return
# Human Label Only
# =========================

input_file=Path(
"step4_final_dataset_outputs/final_alibaba_news_dataset_return_ready.csv"
)

output_dir=Path(
"step7_sentiment_return_outputs"
)

output_dir.mkdir(exist_ok=True)

df=pd.read_csv(input_file)

print("样本量:",len(df))

display(df.head())

# =========================
# 使用人工标签
# =========================

df["sentiment_score"]=(
pd.to_numeric(
df["sentiment_score"],
errors="coerce"
)
)

# 三分类
# 2/3/4

def to_sentiment_3class(x):

    if x in [1,2]:
        return 2

    elif x==3:
        return 3

    elif x in [4,5]:
        return 4

    else:
        return np.nan

df["sentiment_3class"]=(
df["sentiment_score"]
.apply(to_sentiment_3class)
)

# =========================
# return变量
# =========================

return_cols=[

"ret_1d",
"ret_5d",
"ret_21d",

"abret_1d",
"abret_5d",
"abret_21d"

]

# =========================
# 1
# 相关分析
# =========================

corr_results=[]

for col in return_cols:

    temp=df[
    ["sentiment_score",col]
    ].dropna()

    pearson_corr,pearson_p=(
    pearsonr(
    temp["sentiment_score"],
    temp[col]
    )
    )

    spearman_corr,spearman_p=(
    spearmanr(
    temp["sentiment_score"],
    temp[col]
    )
    )

    corr_results.append({

    "return":

    col,

    "N":

    len(temp),

    "pearson_r":

    pearson_corr,

    "pearson_p":

    pearson_p,

    "spearman_r":

    spearman_corr,

    "spearman_p":

    spearman_p

    })

corr_df=pd.DataFrame(
corr_results
)

print()

print(
"========== Correlation =========="
)

display(
corr_df
)

# =========================
# 2
# 分组均值
# =========================

group_mean=(
df
.groupby(
"sentiment_3class"
)
[
return_cols
]
.agg(
[
"count",
"mean",
"median",
"std"
]
)
)

print()

print(
"========== Group Mean =========="
)

display(
group_mean
)

# =========================
# t test
# negative vs positive
# =========================

ttest_results=[]

for col in return_cols:

    neg=(
    df[
    df[
    "sentiment_3class"
    ]==2
    ][col]
    .dropna()
    )

    pos=(
    df[
    df[
    "sentiment_3class"
    ]==4
    ][col]
    .dropna()
    )

    stat,p=ttest_ind(
    neg,
    pos,
    equal_var=False
    )

    ttest_results.append({

    "return":
    col,

    "negative_mean":
    neg.mean(),

    "positive_mean":
    pos.mean(),

    "t_stat":
    stat,

    "p":
    p
    })

ttest_df=pd.DataFrame(
ttest_results
)

print()

print(
"========== Negative vs Positive =========="
)

display(
ttest_df
)

# =========================
# 3
# OLS
# abret~sentiment
# =========================

ols_results=[]

for col in [

"abret_1d",
"abret_5d",
"abret_21d"

]:

    temp=df[
    ["sentiment_score",col]
    ].dropna()

    X=sm.add_constant(
    temp[
    "sentiment_score"
    ]
    )

    y=temp[col]

    model=sm.OLS(
    y,
    X
    ).fit()

    ols_results.append({

    "dependent":

    col,

    "N":

    len(temp),

    "beta":

    model.params[
    "sentiment_score"
    ],

    "p":

    model.pvalues[
    "sentiment_score"
    ],

    "R2":

    model.rsquared

    })

    print()

    print(
    "="*50
    )

    print(
    col
    )

    print(
    model.summary()
    )

ols_df=pd.DataFrame(
ols_results
)

print()

print(
"========== OLS Summary =========="
)

display(
ols_df
)

# =========================
# 保存
# =========================

corr_df.to_csv(

output_dir/
"correlation.csv",

index=False
)

group_mean.to_csv(

output_dir/
"group_mean.csv"
)

ttest_df.to_csv(

output_dir/
"ttest.csv",

index=False
)

ols_df.to_csv(

output_dir/
"ols_summary.csv",

index=False
)

with pd.ExcelWriter(
output_dir/
"step7_sentiment_return.xlsx"
) as writer:

    corr_df.to_excel(
    writer,
    sheet_name="correlation",
    index=False
    )

    group_mean.to_excel(
    writer,
    sheet_name="group_mean"
    )

    ttest_df.to_excel(
    writer,
    sheet_name="ttest",
    index=False
    )

    ols_df.to_excel(
    writer,
    sheet_name="ols"
    )

print()
print(
"Step7完成"
)
print(
output_dir.resolve()
)

样本量: 370


,article_id,date,event_date,title_en,body_en,full_text,sentiment_score,sentiment_reason,source_period,relevance_flag,ticker,ret_1d,ret_5d,ret_21d,abret_1d,abret_5d,abret_21d,sentiment_3class
0,period_1_2025_0200,2025-06-03 06:02:57,2025-06-03,Platform economy stimulates the vitality of sm...,Individual industrial and commercial household...,Platform economy stimulates the vitality of sm...,3,Unrelated to Alibaba company entities,period_1_2025,0,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327,neutral
1,period_1_2025_0199,2025-06-03 08:01:43,2025-06-03,"""Underage movie viewing discount policy 'hide ...","Behind a regular movie ticket, there is a hidd...","""Underage movie viewing discount policy 'hide ...",3,Unrelated to Alibaba company entities,period_1_2025,0,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327,neutral
2,period_1_2025_0198,2025-06-03 08:02:03,2025-06-03,Setting up a scam in the name of borrowing a h...,"Self-proclaimed as the ""red envelope snatching...",Setting up a scam in the name of borrowing a h...,3,Unrelated to Alibaba company entities,period_1_2025,0,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327,neutral
3,period_1_2025_0197,2025-06-03 08:23:36,2025-06-03,Notice on holding the 2024 Academic Annual Con...,Relevant units: Pharmaceutical excipients are ...,Notice on holding the 2024 Academic Annual Con...,3,Unrelated to Alibaba company entities,period_1_2025,0,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327,neutral
4,period_1_2025_0196,2025-06-03 14:13:17,2025-06-03,"Seize the 90-day window period ""Foreign Trade ...",China Business News reporter Li Li reported fr...,"Seize the 90-day window period ""Foreign Trade ...",4,"Taobao's 618 event is being promoted globally,...",period_1_2025,1,9988.HK,0.006146,0.04302,-0.065637,0.000332,0.003292,-0.075327,positive



========== Correlation ==========


,return,N,pearson_r,pearson_p,spearman_r,spearman_p
0,ret_1d,370,0.041990,0.420640,0.083292,0.109706
1,ret_5d,370,0.007254,0.889394,-0.008017,0.877853
2,ret_21d,343,0.109169,0.043331,0.107373,0.046915
3,abret_1d,370,-0.023932,0.646344,0.027888,0.592841
4,abret_5d,370,-0.005765,0.911994,-0.017674,0.734729
5,abret_21d,342,0.086647,0.109704,0.087622,0.105748



========== Group Mean ==========


...


========== Negative vs Positive ==========


,return,negative_mean,positive_mean,t_stat,p
0,ret_1d,-0.007917,-0.001373,-1.347279,0.180875
1,ret_5d,-0.026836,-0.024034,-0.308135,0.758646
2,ret_21d,-0.024041,0.006606,-1.644321,0.104711
3,abret_1d,-0.000386,-0.000342,-0.017315,0.986215
4,abret_5d,-0.010661,-0.014340,0.808390,0.420640
5,abret_21d,-0.011110,-0.008258,-0.280241,0.780113



abret_1d
                            OLS Regression Results                            
Dep. Variable:               abret_1d   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.002
Method:                 Least Squares   F-statistic:                    0.2109
Date:                Tue, 19 May 2026   Prob (F-statistic):              0.646
Time:                        12:26:54   Log-Likelihood:                 1079.7
No. Observations:                 370   AIC:                            -2155.
Df Residuals:                     368   BIC:                            -2148.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            8.674e-06      

,dependent,N,beta,p,R2
0,abret_1d,370,-0.000475,0.646344,0.000573
1,abret_5d,370,-0.000201,0.911994,0.000033
2,abret_21d,342,0.006553,0.109704,0.007508



Step7完成
/Users/sunny/Desktop/Capstone/中期汇报/step7_sentiment_return_outputs


In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import ttest_ind

import statsmodels.api as sm

# =========================
# Step7B
# BERT predicted sentiment
# → return
# =========================

input_file=Path(
"step4_final_dataset_outputs/final_alibaba_news_dataset_return_ready.csv"
)

model_dir=Path(
"step5_bert_sentiment_outputs_clean/best_distilbert_3class_model"
)

output_dir=Path(
"step7B_bert_return_outputs"
)

output_dir.mkdir(exist_ok=True)

df=pd.read_csv(input_file)

print("样本:",len(df))

# =========================
# 加载模型
# =========================

tokenizer=AutoTokenizer.from_pretrained(
model_dir
)

model=(
AutoModelForSequenceClassification
.from_pretrained(
model_dir
)
)

model.eval()

# =========================
# 预测全部新闻
# =========================

texts=(
df["full_text"]
.fillna("")
.astype(str)
.tolist()
)

pred_labels=[]

prob_negative=[]
prob_neutral=[]
prob_positive=[]

BATCH_SIZE=16

for i in range(
0,
len(texts),
BATCH_SIZE
):

    batch_text=(
    texts[
    i:i+BATCH_SIZE
    ]
    )

    inputs=tokenizer(

        batch_text,

        padding=True,

        truncation=True,

        max_length=128,

        return_tensors="pt"
    )

    with torch.no_grad():

        outputs=model(
        **inputs
        )

    logits=outputs.logits

    probs=(
    torch.softmax(
    logits,
    dim=1
    )
    .numpy()
    )

    pred=np.argmax(
    probs,
    axis=1
    )

    pred_labels.extend(
    pred
    )

    prob_negative.extend(
    probs[:,0]
    )

    prob_neutral.extend(
    probs[:,1]
    )

    prob_positive.extend(
    probs[:,2]
    )

# label恢复

id2label={

0:2,
1:3,
2:4

}

df[
"bert_pred_sentiment"
]=[
id2label[x]
for x in pred_labels
]

df[
"prob_2_negative"
]=prob_negative

df[
"prob_3_neutral"
]=prob_neutral

df[
"prob_4_positive"
]=prob_positive

print()

print(
"BERT预测分布"
)

display(

df[
"bert_pred_sentiment"
]
.value_counts()
.sort_index()
)

# =========================
# return
# =========================

return_cols=[

"ret_1d",
"ret_5d",
"ret_21d",

"abret_1d",
"abret_5d",
"abret_21d"

]

# =========================
# correlation
# =========================

corr=[]

for col in return_cols:

    temp=df[
    [
    "bert_pred_sentiment",
    col
    ]
    ].dropna()

    p1,p2=pearsonr(

    temp[
    "bert_pred_sentiment"
    ],

    temp[col]

    )

    s1,s2=spearmanr(

    temp[
    "bert_pred_sentiment"
    ],

    temp[col]

    )

    corr.append({

    "return":
    col,

    "N":
    len(temp),

    "pearson_r":
    p1,

    "pearson_p":
    p2,

    "spearman_r":
    s1,

    "spearman_p":
    s2

    })

corr_df=pd.DataFrame(
corr
)

print()
print(
"Correlation"
)
display(
corr_df
)

# =========================
# group mean
# =========================

group_mean=(
df
.groupby(
"bert_pred_sentiment"
)
[
return_cols
]
.agg(
[
"count",
"mean",
"median",
"std"
]
)
)

print()
print(
"Group Mean"
)

display(
group_mean
)

# =========================
# t test
# =========================

ttest=[]

for col in return_cols:

    neg=(
    df[
    df[
    "bert_pred_sentiment"
    ]==2
    ][col]
    .dropna()
    )

    pos=(
    df[
    df[
    "bert_pred_sentiment"
    ]==4
    ][col]
    .dropna()
    )

    stat,p=ttest_ind(
    neg,
    pos,
    equal_var=False
    )

    ttest.append({

    "return":
    col,

    "negative_mean":
    neg.mean(),

    "positive_mean":
    pos.mean(),

    "t":
    stat,

    "p":
    p

    })

ttest_df=pd.DataFrame(
ttest
)

display(
ttest_df
)

# =========================
# OLS
# =========================

ols=[]

for col in [

"abret_1d",
"abret_5d",
"abret_21d"

]:

    temp=df[
    [
    "bert_pred_sentiment",
    col
    ]
    ].dropna()

    X=sm.add_constant(

    temp[
    "bert_pred_sentiment"
    ]

    )

    y=temp[col]

    model=(
    sm.OLS(
    y,
    X
    )
    .fit()
    )

    print()
    print("="*50)
    print(col)
    print(
    model.summary()
    )

    ols.append({

    "return":
    col,

    "beta":
    model.params[
    "bert_pred_sentiment"
    ],

    "p":
    model.pvalues[
    "bert_pred_sentiment"
    ],

    "R2":
    model.rsquared

    })

ols_df=pd.DataFrame(
ols
)

display(
ols_df
)

# =========================
# 保存
# =========================

with pd.ExcelWriter(
output_dir/
"step7B_bert_return.xlsx"
) as writer:

    df.to_excel(
    writer,
    sheet_name=
    "bert_prediction",
    index=False
    )

    corr_df.to_excel(
    writer,
    sheet_name=
    "correlation",
    index=False
    )

    group_mean.to_excel(
    writer,
    sheet_name=
    "group_mean"
    )

    ttest_df.to_excel(
    writer,
    sheet_name=
    "ttest",
    index=False
    )

    ols_df.to_excel(
    writer,
    sheet_name=
    "ols",
    index=False
    )

print()
print(
"Step7B完成"
)
print(
output_dir.resolve()
)

样本: 370


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


BERT预测分布


bert_pred_sentiment
2     17
3    274
4     79
Name: count, dtype: int64


Correlation


...

ret_1d                               ret_5d            \
                     count      mean    median       std  count      mean   
bert_pred_sentiment                                                         
2                       17 -0.020303 -0.028458  0.022929     17 -0.032637   
3                      274 -0.002680 -0.003643  0.021658    274 -0.022232   
4                       79 -0.001811 -0.003252  0.025808     79 -0.024734   

                                        ret_21d            ...  abret_1d  \
                       median       std   count      mean  ...    median   
bert_pred_sentiment                                        ...             
2                   -0.048099  0.053348      14 -0.033081  ... -0.005442   
3                   -0.025501  0.039870     254 -0.016241  ... -0.002598   
4                   -0.034091  0.048707      75 -0.000704  ...  0.000447   

                              abret_5d                                \
                          std    count      mean    median       std   
bert_pred_sentiment                                                    
2                    0.009367       17 -0.012924 -0.014575  0.025151   
3                    0.012611      274 -0.018152 -0.025873  0.022274   
4                    0.015179       79 -0.014934 -0.019814  0.024849   

                    abret_21d                                
                        count      mean    median       std  
bert_pred_sentiment                                          
2                          14 -0.015240 -0.002266  0.066221  
3                         253 -0.034057 -0.030480  0.045935  
4                          75 -0.009434 -0.001773  0.049442  

[3 rows x 24 columns]

,return,negative_mean,positive_mean,t,p
0,ret_1d,-0.020303,-0.001811,-2.947553,0.006759
1,ret_5d,-0.032637,-0.024734,-0.562434,0.579479
2,ret_21d,-0.033081,-0.000704,-0.989439,0.337237
3,abret_1d,-0.006125,-0.000164,-2.097319,0.042898
4,abret_5d,-0.012924,-0.014934,0.299540,0.767193
5,abret_21d,-0.015240,-0.009434,-0.312218,0.758953



abret_1d
                            OLS Regression Results                            
Dep. Variable:               abret_1d   R-squared:                       0.006
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     2.263
Date:                Tue, 19 May 2026   Prob (F-statistic):              0.133
Time:                        12:45:49   Log-Likelihood:                 1080.7
No. Observations:                 370   AIC:                            -2157.
Df Residuals:                     368   BIC:                            -2150.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -

,return,beta,p,R2
0,abret_1d,0.002126,0.133331,0.006113
1,abret_5d,0.001260,0.612481,0.000698
2,abret_21d,0.015450,0.004846,0.023105



Step7B完成
/Users/sunny/Desktop/Capstone/中期汇报/step7B_bert_return_outputs


In [22]:
!pip install transformers datasets accelerate scikit-learn openpyxl -q

In [23]:
import pandas as pd
import numpy as np
from pathlib import Path
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# =========================
# Step 8：End-to-End Return Prediction
# full_text → abret_21d
# =========================

input_file = Path("step4_final_dataset_outputs/final_alibaba_news_dataset_return_ready.csv")

output_dir = Path("step8_end_to_end_return_outputs")
output_dir.mkdir(exist_ok=True)

df = pd.read_csv(input_file)

print("原始样本量:", len(df))
print("GPU 是否可用:", torch.cuda.is_available())

# =========================
# 1. 设置预测目标
# =========================

TARGET_COL = "abret_21d"

df_model = df.dropna(subset=["full_text", TARGET_COL]).copy()
df_model["full_text"] = df_model["full_text"].astype(str).str.strip()
df_model = df_model[df_model["full_text"] != ""].copy()

df_model[TARGET_COL] = pd.to_numeric(df_model[TARGET_COL], errors="coerce")
df_model = df_model.dropna(subset=[TARGET_COL]).copy()

print("可用于端到端回报预测的样本量:", len(df_model))
print("预测目标:", TARGET_COL)

display(df_model[[TARGET_COL]].describe())

# =========================
# 2. 按时间排序后划分 train / val / test
# 金融预测建议用时间切分，而不是随机切分
# =========================

df_model["event_date"] = pd.to_datetime(df_model["event_date"], errors="coerce")
df_model = df_model.sort_values("event_date").reset_index(drop=True)

n = len(df_model)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train = df_model.iloc[:train_end].copy()
val = df_model.iloc[train_end:val_end].copy()
test = df_model.iloc[val_end:].copy()

print("\n时间切分结果:")
print("Train:", len(train), train["event_date"].min(), "到", train["event_date"].max())
print("Val:", len(val), val["event_date"].min(), "到", val["event_date"].max())
print("Test:", len(test), test["event_date"].min(), "到", test["event_date"].max())

# =========================
# 3. 构造 HuggingFace Dataset
# =========================

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 回归任务要求 label 是 float
for x in [train, val, test]:
    x["label"] = x[TARGET_COL].astype(float)

keep_cols = [
    "article_id",
    "full_text",
    TARGET_COL,
    "label"
]

train_ds = Dataset.from_pandas(train[keep_cols])
val_ds = Dataset.from_pandas(val[keep_cols])
test_ds = Dataset.from_pandas(test[keep_cols])

def tokenize_function(examples):
    return tokenizer(
        examples["full_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================
# 4. 建立 DistilBERT 回归模型
# num_labels=1 表示回归
# =========================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="regression"
)

# =========================
# 5. 评价指标
# =========================

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    preds = predictions.reshape(-1)
    labels = labels.reshape(-1)

    mse = mean_squared_error(labels, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(labels, preds)

    if len(np.unique(preds)) > 1 and len(np.unique(labels)) > 1:
        pearson_r, pearson_p = pearsonr(labels, preds)
        spearman_r, spearman_p = spearmanr(labels, preds)
    else:
        pearson_r, pearson_p = np.nan, np.nan
        spearman_r, spearman_p = np.nan, np.nan

    directional_accuracy = np.mean(np.sign(labels) == np.sign(preds))

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "pearson_r": pearson_r,
        "spearman_r": spearman_r,
        "directional_accuracy": directional_accuracy
    }

# =========================
# 6. 训练
# =========================

training_args = TrainingArguments(
    output_dir=str(output_dir / "training_checkpoints"),
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=4,
    weight_decay=0.01,
    report_to="none",
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# =========================
# 7. 测试集预测
# =========================

pred_output = trainer.predict(test_ds)

y_true = pred_output.label_ids.reshape(-1)
y_pred = pred_output.predictions.reshape(-1)

mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_true, y_pred)

pearson_r, pearson_p = pearsonr(y_true, y_pred)
spearman_r, spearman_p = spearmanr(y_true, y_pred)

directional_accuracy = np.mean(np.sign(y_true) == np.sign(y_pred))

metrics_summary = pd.DataFrame([{
    "model": MODEL_NAME,
    "task": f"End-to-end return prediction: full_text → {TARGET_COL}",
    "target": TARGET_COL,
    "train_size": len(train),
    "validation_size": len(val),
    "test_size": len(test),
    "mse": mse,
    "rmse": rmse,
    "mae": mae,
    "pearson_r": pearson_r,
    "pearson_p": pearson_p,
    "spearman_r": spearman_r,
    "spearman_p": spearman_p,
    "directional_accuracy": directional_accuracy
}])

print("\n========== Test Metrics ==========")
display(metrics_summary)

# =========================
# 8. 保存测试集预测结果
# =========================

test_predictions = test.copy().reset_index(drop=True)

test_predictions["true_return"] = y_true
test_predictions["pred_return"] = y_pred
test_predictions["prediction_error"] = test_predictions["pred_return"] - test_predictions["true_return"]
test_predictions["abs_error"] = test_predictions["prediction_error"].abs()
test_predictions["true_direction"] = np.sign(test_predictions["true_return"])
test_predictions["pred_direction"] = np.sign(test_predictions["pred_return"])
test_predictions["correct_direction"] = (
    test_predictions["true_direction"] == test_predictions["pred_direction"]
).astype(int)

prediction_cols = [
    "article_id",
    "date",
    "event_date",
    "title_en",
    "body_en",
    "full_text",
    "sentiment_score",
    "source_period",
    "relevance_flag",
    TARGET_COL,
    "true_return",
    "pred_return",
    "prediction_error",
    "abs_error",
    "true_direction",
    "pred_direction",
    "correct_direction"
]

prediction_cols = [c for c in prediction_cols if c in test_predictions.columns]
test_predictions_clean = test_predictions[prediction_cols].copy()

print("\n========== Test Predictions Preview ==========")
display(test_predictions_clean.head())

# =========================
# 9. Baseline：永远预测训练集平均收益
# =========================

baseline_pred = np.repeat(train[TARGET_COL].mean(), len(test))
baseline_mse = mean_squared_error(y_true, baseline_pred)
baseline_rmse = np.sqrt(baseline_mse)
baseline_mae = mean_absolute_error(y_true, baseline_pred)
baseline_directional_accuracy = np.mean(np.sign(y_true) == np.sign(baseline_pred))

baseline_summary = pd.DataFrame([{
    "model": "Mean Return Baseline",
    "target": TARGET_COL,
    "mse": baseline_mse,
    "rmse": baseline_rmse,
    "mae": baseline_mae,
    "directional_accuracy": baseline_directional_accuracy
}])

print("\n========== Baseline Metrics ==========")
display(baseline_summary)

comparison_summary = pd.concat([
    metrics_summary[["model", "target", "mse", "rmse", "mae", "directional_accuracy"]],
    baseline_summary[["model", "target", "mse", "rmse", "mae", "directional_accuracy"]]
], ignore_index=True)

print("\n========== Model vs Baseline ==========")
display(comparison_summary)

# =========================
# 10. 保存结果
# =========================

metrics_summary.to_csv(
    output_dir / f"bert_return_prediction_metrics_{TARGET_COL}.csv",
    index=False,
    encoding="utf-8-sig"
)

baseline_summary.to_csv(
    output_dir / f"baseline_metrics_{TARGET_COL}.csv",
    index=False,
    encoding="utf-8-sig"
)

comparison_summary.to_csv(
    output_dir / f"model_vs_baseline_{TARGET_COL}.csv",
    index=False,
    encoding="utf-8-sig"
)

test_predictions_clean.to_csv(
    output_dir / f"bert_return_test_predictions_{TARGET_COL}.csv",
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(output_dir / f"step8_end_to_end_return_prediction_{TARGET_COL}.xlsx", engine="openpyxl") as writer:
    metrics_summary.to_excel(writer, sheet_name="bert_metrics", index=False)
    baseline_summary.to_excel(writer, sheet_name="baseline_metrics", index=False)
    comparison_summary.to_excel(writer, sheet_name="comparison", index=False)
    test_predictions_clean.to_excel(writer, sheet_name="test_predictions", index=False)

readme_text = f"""
Step 8 End-to-End Return Prediction
===================================

Task
----
Input:
- full_text = news title + news body

Target:
- {TARGET_COL}

Model
-----
- {MODEL_NAME}
- Regression head
- The model directly predicts future abnormal return from news text.
- This step does not use sentiment_score as input.

Data Split
----------
A time-based split is used:
- first 60% observations: training set
- next 20% observations: validation set
- last 20% observations: test set

This avoids look-ahead bias and is more appropriate for financial prediction.

Output Files
------------
1. bert_return_prediction_metrics_{TARGET_COL}.csv
   Test-set metrics for the BERT regression model.

2. baseline_metrics_{TARGET_COL}.csv
   Metrics for the mean-return baseline.

3. model_vs_baseline_{TARGET_COL}.csv
   Side-by-side comparison between BERT and baseline.

4. bert_return_test_predictions_{TARGET_COL}.csv
   Article-level prediction results.

Metrics
-------
- MSE: mean squared error
- RMSE: square root of MSE
- MAE: mean absolute error
- Pearson r: linear correlation between true and predicted returns
- Spearman r: rank correlation between true and predicted returns
- Directional accuracy: whether the model predicts the correct sign of return

Interpretation
--------------
If BERT outperforms the mean-return baseline and has positive correlation with actual returns,
then the text contains useful information for predicting future abnormal returns.

If BERT does not outperform the baseline, this suggests that direct return prediction from small-sample news text is difficult.
"""

with open(output_dir / "README_step8_outputs.txt", "w", encoding="utf-8") as f:
    f.write(readme_text)

print("\nStep 8 完成，结果已保存到：")
print(output_dir.resolve())

原始样本量: 370
GPU 是否可用: False
可用于端到端回报预测的样本量: 342
预测目标: abret_21d


,abret_21d
count,342.000000
mean,-0.027887
std,0.048649
min,-0.109377
25%,-0.075293
50%,-0.024734
75%,0.020202
max,0.063379



时间切分结果:
Train: 205 2025-06-03 00:00:00 到 2026-02-12 00:00:00
Val: 68 2026-02-12 00:00:00 到 2026-03-19 00:00:00
Test: 69 2026-03-20 00:00:00 到 2026-04-16 00:00:00


Map:   0%|          | 0/205 [00:00<?, ? examples/s]

Map:   0%|          | 0/68 [00:00<?, ? examples/s]

...


========== Baseline Metrics ==========


,model,target,mse,rmse,mae,directional_accuracy
0,Mean Return Baseline,abret_21d,0.006243,0.079014,0.076188,0.115942



========== Model vs Baseline ==========


,model,target,mse,rmse,mae,directional_accuracy
0,distilbert-base-uncased,abret_21d,0.007462,0.086384,0.083510,0.115942
1,Mean Return Baseline,abret_21d,0.006243,0.079014,0.076188,0.115942



Step 8 完成，结果已保存到：
/Users/sunny/Desktop/Capstone/中期汇报/step8_end_to_end_return_outputs


In [25]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# Step10：最终横向比较
# =========================

output_dir = Path("step10_final_comparison_outputs")
output_dir.mkdir(exist_ok=True)

# =========================
# 1. 输入文件路径
# =========================

step7a_file = Path("step7_sentiment_return_outputs/step7_sentiment_return.xlsx")
step7b_file = Path("step7B_bert_return_outputs/step7B_bert_return.xlsx")

step8_compare_file = Path("step8_end_to_end_return_outputs/model_vs_baseline_abret_21d.csv")
step8_metrics_file = Path("step8_end_to_end_return_outputs/bert_return_prediction_metrics_abret_21d.csv")

step5_metrics_file = Path("step5_bert_sentiment_outputs_clean/bert_3class_metrics_summary.csv")

TARGET = "abret_21d"

# =========================
# 2. 辅助函数
# =========================

def get_corr_row(corr_df, target=TARGET):
    if "return" not in corr_df.columns:
        raise ValueError("correlation table must contain column: return")
    row = corr_df[corr_df["return"] == target]
    if len(row) == 0:
        raise ValueError(f"Cannot find {target} in correlation table.")
    return row.iloc[0]


def get_ols_row(ols_df, target=TARGET):
    """
    兼容两种格式：
    Step7A: dependent, N, beta, p, R2
    Step7B: return, beta, p, R2
    """
    if "dependent" in ols_df.columns:
        row = ols_df[ols_df["dependent"] == target]
    elif "return" in ols_df.columns:
        row = ols_df[ols_df["return"] == target]
    else:
        raise ValueError("OLS table must contain either 'dependent' or 'return' column.")
    
    if len(row) == 0:
        raise ValueError(f"Cannot find {target} in OLS table.")
    
    return row.iloc[0]


def safe_get(row, col, default=np.nan):
    return row[col] if col in row.index else default


def significance_label(p):
    if pd.isna(p):
        return "N/A"
    elif p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.1:
        return "*"
    else:
        return "Not significant"


# =========================
# 3. 读取 Step7A：人工 sentiment → return
# =========================

human_corr = pd.read_excel(step7a_file, sheet_name="correlation")
human_ols = pd.read_excel(step7a_file, sheet_name="ols")

human_corr_21d = get_corr_row(human_corr, TARGET)
human_ols_21d = get_ols_row(human_ols, TARGET)

print("Step7A human sentiment loaded.")
display(human_corr)
display(human_ols)


# =========================
# 4. 读取 Step7B：BERT predicted sentiment → return
# =========================

bert_corr = pd.read_excel(step7b_file, sheet_name="correlation")
bert_ols = pd.read_excel(step7b_file, sheet_name="ols")

bert_corr_21d = get_corr_row(bert_corr, TARGET)
bert_ols_21d = get_ols_row(bert_ols, TARGET)

print("\nStep7B BERT sentiment loaded.")
display(bert_corr)
display(bert_ols)


# =========================
# 5. 读取 Step8：End-to-End return prediction
# =========================

e2e_metrics = pd.read_csv(step8_metrics_file)
e2e_compare = pd.read_csv(step8_compare_file)

e2e_row = e2e_metrics.iloc[0]

print("\nStep8 end-to-end loaded.")
display(e2e_metrics)
display(e2e_compare)


# =========================
# 6. 读取 Step5：BERT sentiment classifier performance
# =========================

if step5_metrics_file.exists():
    step5_metrics = pd.read_csv(step5_metrics_file)
    print("\nStep5 BERT sentiment classifier loaded.")
    display(step5_metrics)
else:
    step5_metrics = pd.DataFrame()
    print("\nStep5 metrics file not found. Skip Step5 summary.")


# =========================
# 7. 构造核心横向比较表
# =========================

comparison_core = pd.DataFrame([
    {
        "method": "Human sentiment score",
        "path": "Human sentiment → abret_21d",
        "input": "manual sentiment_score",
        "target": TARGET,
        "pearson_r": safe_get(human_corr_21d, "pearson_r"),
        "pearson_p": safe_get(human_corr_21d, "pearson_p"),
        "spearman_r": safe_get(human_corr_21d, "spearman_r"),
        "spearman_p": safe_get(human_corr_21d, "spearman_p"),
        "ols_beta": safe_get(human_ols_21d, "beta"),
        "ols_p": safe_get(human_ols_21d, "p"),
        "ols_r2": safe_get(human_ols_21d, "R2"),
        "rmse": np.nan,
        "mae": np.nan,
        "directional_accuracy": np.nan,
        "main_conclusion": "Weak positive relationship; not statistically significant for 21-day abnormal return."
    },
    {
        "method": "BERT predicted sentiment",
        "path": "BERT predicted sentiment → abret_21d",
        "input": "BERT predicted sentiment class",
        "target": TARGET,
        "pearson_r": safe_get(bert_corr_21d, "pearson_r"),
        "pearson_p": safe_get(bert_corr_21d, "pearson_p"),
        "spearman_r": safe_get(bert_corr_21d, "spearman_r"),
        "spearman_p": safe_get(bert_corr_21d, "spearman_p"),
        "ols_beta": safe_get(bert_ols_21d, "beta"),
        "ols_p": safe_get(bert_ols_21d, "p"),
        "ols_r2": safe_get(bert_ols_21d, "R2"),
        "rmse": np.nan,
        "mae": np.nan,
        "directional_accuracy": np.nan,
        "main_conclusion": "Stronger positive and statistically significant relationship with 21-day abnormal return."
    },
    {
        "method": "End-to-end BERT regression",
        "path": "News text → abret_21d",
        "input": "full_text",
        "target": TARGET,
        "pearson_r": safe_get(e2e_row, "pearson_r"),
        "pearson_p": safe_get(e2e_row, "pearson_p"),
        "spearman_r": safe_get(e2e_row, "spearman_r"),
        "spearman_p": safe_get(e2e_row, "spearman_p"),
        "ols_beta": np.nan,
        "ols_p": np.nan,
        "ols_r2": np.nan,
        "rmse": safe_get(e2e_row, "rmse"),
        "mae": safe_get(e2e_row, "mae"),
        "directional_accuracy": safe_get(e2e_row, "directional_accuracy"),
        "main_conclusion": "Direct return prediction shows weak correlation but does not outperform the mean-return baseline."
    }
])

comparison_core["pearson_significance"] = comparison_core["pearson_p"].apply(significance_label)
comparison_core["ols_significance"] = comparison_core["ols_p"].apply(significance_label)

print("\n========== Core Comparison ==========")
display(comparison_core)


# =========================
# 8. End-to-End vs Baseline 对比
# =========================

print("\n========== End-to-End vs Mean Baseline ==========")
display(e2e_compare)

baseline_row = e2e_compare[
    e2e_compare["model"].astype(str).str.contains("Baseline", case=False, na=False)
]

bert_e2e_row = e2e_compare[
    ~e2e_compare["model"].astype(str).str.contains("Baseline", case=False, na=False)
]

if len(baseline_row) > 0:
    baseline_rmse = baseline_row.iloc[0]["rmse"]
    baseline_mae = baseline_row.iloc[0]["mae"]
    baseline_da = baseline_row.iloc[0]["directional_accuracy"]
else:
    baseline_rmse = np.nan
    baseline_mae = np.nan
    baseline_da = np.nan

if len(bert_e2e_row) > 0:
    bert_e2e_rmse = bert_e2e_row.iloc[0]["rmse"]
    bert_e2e_mae = bert_e2e_row.iloc[0]["mae"]
    bert_e2e_da = bert_e2e_row.iloc[0]["directional_accuracy"]
else:
    bert_e2e_rmse = safe_get(e2e_row, "rmse")
    bert_e2e_mae = safe_get(e2e_row, "mae")
    bert_e2e_da = safe_get(e2e_row, "directional_accuracy")


# =========================
# 9. 生成 report-ready summary
# =========================

report_summary = comparison_core[
    [
        "method",
        "path",
        "input",
        "target",
        "pearson_r",
        "pearson_p",
        "pearson_significance",
        "spearman_r",
        "spearman_p",
        "ols_beta",
        "ols_p",
        "ols_significance",
        "ols_r2",
        "rmse",
        "mae",
        "directional_accuracy",
        "main_conclusion"
    ]
].copy()

print("\n========== Report-ready Summary ==========")
display(report_summary)


# =========================
# 10. 自动生成文字解释
# =========================

human_r = safe_get(human_corr_21d, "pearson_r")
human_p = safe_get(human_corr_21d, "pearson_p")
human_beta = safe_get(human_ols_21d, "beta")
human_beta_p = safe_get(human_ols_21d, "p")

bert_r = safe_get(bert_corr_21d, "pearson_r")
bert_p = safe_get(bert_corr_21d, "pearson_p")
bert_beta = safe_get(bert_ols_21d, "beta")
bert_beta_p = safe_get(bert_ols_21d, "p")

e2e_r = safe_get(e2e_row, "pearson_r")
e2e_p = safe_get(e2e_row, "pearson_p")
e2e_rmse = safe_get(e2e_row, "rmse")
e2e_mae = safe_get(e2e_row, "mae")
e2e_da = safe_get(e2e_row, "directional_accuracy")

conclusion_text = f"""
Step10 Final Horizontal Comparison
==================================

Target variable:
- {TARGET}

Compared Methods
----------------
1. Human sentiment score → 21-day abnormal return
2. BERT predicted sentiment → 21-day abnormal return
3. End-to-end BERT regression: news text → 21-day abnormal return

Main Findings
-------------

1. Human sentiment score shows only weak explanatory power.
   - Pearson r = {human_r:.4f}
   - Pearson p-value = {human_p:.4f}
   - OLS beta = {human_beta:.4f}
   - OLS p-value = {human_beta_p:.4f}

   Interpretation:
   Human sentiment is positively related to future 21-day abnormal return,
   but the relationship is not statistically significant.

2. BERT-predicted sentiment shows stronger explanatory power.
   - Pearson r = {bert_r:.4f}
   - Pearson p-value = {bert_p:.4f}
   - OLS beta = {bert_beta:.4f}
   - OLS p-value = {bert_beta_p:.4f}

   Interpretation:
   BERT-predicted sentiment has a stronger and statistically significant
   relationship with 21-day abnormal return.

3. End-to-end BERT return regression is less stable.
   - Pearson r = {e2e_r:.4f}
   - Pearson p-value = {e2e_p:.4f}
   - RMSE = {e2e_rmse:.4f}
   - MAE = {e2e_mae:.4f}
   - Directional accuracy = {e2e_da:.4f}

   Compared with the mean-return baseline:
   - BERT end-to-end RMSE = {bert_e2e_rmse:.4f}
   - Baseline RMSE = {baseline_rmse:.4f}
   - BERT end-to-end MAE = {bert_e2e_mae:.4f}
   - Baseline MAE = {baseline_mae:.4f}
   - BERT end-to-end directional accuracy = {bert_e2e_da:.4f}
   - Baseline directional accuracy = {baseline_da:.4f}

Overall Conclusion
------------------
The results suggest that news text contains some investment-relevant information,
but this information is better captured through a sentiment-based intermediate representation
than through direct end-to-end return regression.

Among the tested approaches, BERT-predicted sentiment provides the strongest evidence
of return relevance for 21-day abnormal returns. Human sentiment shows a weaker positive
relationship, while direct end-to-end BERT return prediction does not clearly outperform
the mean-return baseline.
"""

print("\n========== Auto-generated Interpretation ==========")
print(conclusion_text)


# =========================
# 11. 保存结果
# =========================

comparison_core.to_csv(
    output_dir / "step10_core_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

report_summary.to_csv(
    output_dir / "step10_report_ready_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

e2e_compare.to_csv(
    output_dir / "step10_end_to_end_vs_baseline.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(output_dir / "step10_final_interpretation.txt", "w", encoding="utf-8") as f:
    f.write(conclusion_text)

with pd.ExcelWriter(output_dir / "step10_final_comparison.xlsx", engine="openpyxl") as writer:
    comparison_core.to_excel(writer, sheet_name="core_comparison", index=False)
    report_summary.to_excel(writer, sheet_name="report_summary", index=False)
    e2e_compare.to_excel(writer, sheet_name="e2e_vs_baseline", index=False)
    
    if len(step5_metrics) > 0:
        step5_metrics.to_excel(writer, sheet_name="bert_sentiment_metrics", index=False)

print("\nStep10 完成，结果已保存到：")
print(output_dir.resolve())

Step7A human sentiment loaded.


,return,N,pearson_r,pearson_p,spearman_r,spearman_p
0,ret_1d,370,0.041990,0.420640,0.083292,0.109706
1,ret_5d,370,0.007254,0.889394,-0.008017,0.877853
2,ret_21d,343,0.109169,0.043331,0.107373,0.046915
3,abret_1d,370,-0.023932,0.646344,0.027888,0.592841
4,abret_5d,370,-0.005765,0.911994,-0.017674,0.734729
5,abret_21d,342,0.086647,0.109704,0.087622,0.105748


,Unnamed: 0,dependent,N,beta,p,R2
0,0,abret_1d,370,-0.000475,0.646344,0.000573
1,1,abret_5d,370,-0.000201,0.911994,0.000033
2,2,abret_21d,342,0.006553,0.109704,0.007508



Step7B BERT sentiment loaded.


,return,N,pearson_r,pearson_p,spearman_r,spearman_p
0,ret_1d,370,0.099943,0.054763,0.091825,0.077726
1,ret_5d,370,0.005557,0.915160,-0.007323,0.888356
2,ret_21d,343,0.087547,0.105535,0.086584,0.109441
3,abret_1d,370,0.078183,0.133331,0.075705,0.146120
4,abret_5d,370,0.026418,0.612481,0.028590,0.583560
5,abret_21d,342,0.152003,0.004846,0.162151,0.002633


...


========== End-to-End vs Mean Baseline ==========


,model,target,mse,rmse,mae,directional_accuracy
0,distilbert-base-uncased,abret_21d,0.007462,0.086384,0.083510,0.115942
1,Mean Return Baseline,abret_21d,0.006243,0.079014,0.076188,0.115942



========== Report-ready Summary ==========


,method,path,input,target,pearson_r,pearson_p,pearson_significance,spearman_r,spearman_p,ols_beta,ols_p,ols_significance,ols_r2,rmse,mae,directional_accuracy,main_conclusion
0,Human sentiment score,Human sentiment → abret_21d,manual sentiment_score,abret_21d,0.086647,0.109704,Not significant,0.087622,0.105748,0.006553,0.109704,Not significant,0.007508,NaN,NaN,NaN,Weak positive relationship; not statistically ...
1,BERT predicted sentiment,BERT predicted sentiment → abret_21d,BERT predicted sentiment class,abret_21d,0.152003,0.004846,***,0.162151,0.002633,0.015450,0.004846,***,0.023105,NaN,NaN,NaN,Stronger positive and statistically significan...
2,End-to-end BERT regression,News text → abret_21d,full_text,abret_21d,0.221274,0.067672,*,0.205323,0.090557,NaN,NaN,N/A,NaN,0.086384,0.08351,0.115942,Direct return prediction shows weak correlatio...



========== Auto-generated Interpretation ==========

Step10 Final Horizontal Comparison

Target variable:
- abret_21d

Compared Methods
----------------
1. Human sentiment score → 21-day abnormal return
2. BERT predicted sentiment → 21-day abnormal return
3. End-to-end BERT regression: news text → 21-day abnormal return

Main Findings
-------------

1. Human sentiment score shows only weak explanatory power.
   - Pearson r = 0.0866
   - Pearson p-value = 0.1097
   - OLS beta = 0.0066
   - OLS p-value = 0.1097

   Interpretation:
   Human sentiment is positively related to future 21-day abnormal return,
   but the relationship is not statistically significant.

2. BERT-predicted sentiment shows stronger explanatory power.
   - Pearson r = 0.1520
   - Pearson p-value = 0.0048
   - OLS beta = 0.0154
   - OLS p-value = 0.0048

   Interpretation:
   BERT-predicted sentiment has a stronger and statistically significant
   relationship with 21-day abnormal return.

3. End-to-end BERT return 